# BTCUSDT EDA Parameter Optimization Pipeline

> **Archived notebook**: `/Workspace/Users/aaronng.dev1@gmail.com/BTCUSDT Direction Prediction AutoML` (ID `1710400141691875`)  
> **Source table**: `smart-trading-dev.01_bronze.raw_aggtrades` — 52,471,665 deduplicated ticks, Feb 2026 (28 days)  
> **Utility package**: `btcusdt_utils/` v0.3.0 — subpackage architecture with 4 domain subpackages (`features/`, `scoring/`, `evaluation/`, `training/`) + 6 flat modules  
> **Current best ROC-AUC**: 0.519 (LogisticRegression) — effectively random chance  
> **Baseline to beat**: "Last TB Label" persistence at 0.708 AUC / 70.9% accuracy  
> **Price range**: $60,000 – $79,424 &nbsp;|&nbsp; **Avg trades/min**: 1,301 &nbsp;|&nbsp; **Regime event**: sharp drawdown Feb 4–6

---

## Diagnosis Summary

| # | Root Cause | AFML Ref | Impact | Addressed |
|---|---|---|---|---|
| 1 | Non-stationary features (SMA, EMA, BB, MACD at price-scale with unit roots) | Ch. 5 | Tree splits on train prices are meaningless in test — feature space drift | ✅ Phase 3 (FFD) + Phase 4 (stationary-by-construction) |
| 2 | No CUSUM filter — uniform 1-min bars treat noise-dominated and crash minutes equally | Ch. 2 | \~60–70% of 40K bars are low-information noise diluting signal | ✅ Phase 1 (volume bars replace time bars) |
| 3 | Hardcoded TB params (PT=SL=1σ, h=30, span=50) — 80%+ labels resolve in 1–2 bars | Ch. 3 | Extreme label autocorrelation → "Last TB Label" baseline dominates | ✅ Phase 2 (optimized: PT=SL=0.5σ, h=480, span=20) |
| 4 | No stationarity testing (ADF/KPSS) in EDA | Ch. 5 | Non-stationary features pass all quality checks undetected | ✅ Phase 3 (full ADF/KPSS audit) |
| 5 | No sample weighting — overlapping TB labels all receive equal weight | Ch. 4 | Effective independent sample size \~800, not 24K | ⬜ Phase 5 |
| 6 | Unoptimized indicator windows — daily-equity defaults on 1-min crypto | — | Flat feature importance (0.012–0.025); no feature captures signal | ✅ Phase 4 (149 features across 3 tiers, window sweep) |
| 7 | No purging/embargo at train/test boundaries | Ch. 7 | First 30 test labels overlap with training data | ⬜ Phase 6 |

---

## Package Architecture: `btcusdt_utils/` v0.3.0

Subpackage architecture — Polars-first, ≤200 lines/file, frozen dataclass configs per module:

```
btcusdt_utils/
├── __init__.py              # Top-level re-exports (v0.3.0, backward compat)
├── constants.py             # CATALOG, BRONZE, SILVER, GOLD, RAW_AGGTRADES
├── viz.py                   # Plotly dark theme palette, DARK_LAYOUT, dark_fig()
├── bars.py                  # build_time_bars, build_threshold_bars, cusum_filter
├── labeling.py              # apply_triple_barrier (vectorized wavefront scan)
├── quality.py               # BarQuality, LabelQuality, compute_bar/label_quality
├── stationarity.py          # adf_test, kpss_test, ffd, find_min_ffd
│
├── features/                # Subpackage: feature engineering
│   ├── __init__.py          # Re-exports + legacy constants (FEATURE_COLS etc.)
│   ├── _config.py           # TechnicalConfig, MicrostructureConfig, EntropyConfig, StructuralConfig
│   ├── technical.py         # Stationary-by-construction TA (sma_dist, ema_dist, RSI, MACD, BB, ATR)
│   ├── microstructure.py    # Tick→bar aggregation, VPIN, Kyle's Lambda, Amihud, Roll/CS spreads
│   ├── entropy.py           # Shannon entropy, Lempel-Ziv complexity
│   └── structural.py        # CUSUM statistic, rolling ADF, martingale difference
│
├── scoring/                 # Subpackage: feature scoring
│   ├── __init__.py
│   ├── _config.py           # ScoringConfig
│   ├── mi.py                # Mutual information + permutation noise floor
│   └── univariate.py        # Single-feature LightGBM AUC
│
├── evaluation/              # Subpackage: model evaluation (deferred import)
│   ├── __init__.py
│   ├── metrics.py           # compute_metrics, BaselineResult, compute_baselines
│   ├── comparison.py        # build_comparison_table
│   └── plots.py             # ROC curves, feature importance (requires xgboost)
│
└── training/                # Subpackage: model training (deferred import)
    ├── __init__.py
    ├── _config.py           # ModelConfig, TrainResult
    ├── objectives.py        # Optuna objective factories
    └── pipeline.py          # train_and_evaluate (requires optuna + xgboost)
```

**Import pattern** (Cell 4):  
```python
import sys; sys.path.insert(0, "/Workspace/Users/aaronng.dev1@gmail.com")
from btcusdt_utils import *
```

**Note**: `evaluation/` and `training/` are deferred imports (wrapped in `try/except` in `__init__.py`) — they require `optuna` and `xgboost` which are installed in Phase 7.

---

## Phase Map & Status

```
✅ Phase 0: Infrastructure                → Cells 2–5   (installs, package import, 52.5M ticks in Polars)
✅ Phase 1: Sampling Strategy Sweep        → Cells 6–9   (time_5m bars, 8,064 bars, |ACF(1)|≈0.003)
✅ Phase 2: Triple Barrier Optimization    → Cells 10–12 (295 configs, score=0.4655, 7,584 labels)
✅ Phase 3: Feature Stationarity & Fracdiff→ Cells 13–16 (27 features audited, 9 FFD-transformed, mean d*=0.117)
✅ Phase 4: Feature Engineering & Scoring  → Cells 17–22 (3 tiers, 149→132 scored, 0 above MI noise floor)
⬜ Phase 5: Sample Weighting               → AFML Ch. 4  (uniqueness weights, seq. bootstrap)
⬜ Phase 6: Purged K-Fold CV               → AFML Ch. 7  (embargo + purging)
⬜ Phase 7: Retrain & Compare              → final model vs 0.519 baseline
```

---

## Phase 0 — Infrastructure &ensp; ✅ COMPLETE

**Cells**: 2 (install), 3 (restart), 4 (imports from `btcusdt_utils`), 5 (load data)  
**Output**: `df_ticks` — Polars DataFrame, 52,471,665 × 5  
&emsp; Columns: `transact_timestamp`, `price_d`, `qty_d`, `dollar_d`, `is_buyer_maker`  
&emsp; Memory: \~1,608 MB  
**Helpers**: All functions loaded from `btcusdt_utils` package (see Package Architecture above)  
**Third-party imports**: `numpy`, `polars`, `plotly`, `pyspark.sql`, `scipy.stats`, `mlflow`

---

## Phase 1 — Sampling Strategy Sweep &ensp; ✅ COMPLETE

> **Objective**: Find the bar-sampling scheme that produces the most IID-like returns.  
> **Why first**: Everything downstream (labels, features, weights) depends on how we slice the price series.  
> **Input**: `df_ticks` (52.5M rows from Phase 0)  
> **Output**: `df_bars_best` (Polars DataFrame of OHLCV bars), `best_bar_config` (dict)

**Cells**: 6 (bar candidates), 7 (quality eval + MLflow), 8 (CUSUM overlay), 9 (viz + decision)  
**MLflow experiment**: `/Users/aaronng.dev1@gmail.com/BTCUSDT_Phase1_BarSampling`

### Results

| Metric | Value |
|---|---|
| Selected bar type | `time_5m` — 5-minute time bars |
| Bar count | 8,064 |
| \|ACF(1)\| | **0.00263** — near-zero autocorrelation |
| Ljung-Box Q₁₀ p-value | > 0.05 — fails to reject IID null |
| Return σ | 0.00215 |
| CUSUM filter | Not adopted — all thresholds increased ACF while reducing n below 5K |

**Sweep summary** (15 configs tested: 4 time, 4 volume, 4 dollar, 3 tick):  
&emsp; Median per-min calibration: 10.08 BTC volume, $698,740 dollar, 858 ticks  
&emsp; 13 of 15 configs met n ≥ 5,000; time\_15m and time\_30m excluded  
&emsp; Top 3: volume\_10x (0.00047), time\_5m (0.00263), dollar\_5x (0.00704)  

**Key insight**: Information-driven bars (volume) produce more IID-like returns than fixed-time bars on 1-min BTC data, consistent with AFML Ch. 2 theory.

### Cell 6 — Compute Bar Candidates

**6a. Time bars** — intervals \{1, 5, 15, 30\} min   
**6b. Volume bars** — thresholds at \{1×, 2×, 5×, 10×\} of median per-minute volume  
**6c. Dollar bars** — thresholds at \{1×, 2×, 5×, 10×\} of median per-minute dollar volume  
**6d. Tick bars** — thresholds at \{1×, 2×, 5×\} of median per-minute tick count  

### Cell 7 — Evaluate Bar Quality

| Metric | Function | Target |
|---|---|---|
| Jarque-Bera statistic | `sp_stats.jarque_bera` | Lower = more Gaussian |
| Ljung-Box Q at lag 10 | `acorr_ljungbox` | p > 0.05 = no significant ACF |
| ACF(1) | `np.corrcoef(ret[:-1], ret[1:])` | \|ACF(1)\| < 0.05 |
| Sample count | `len(returns)` | ≥ 5,000 for ML |

### Cell 8 — CUSUM Filter Overlay

Sweep thresholds: h ∈ \{1σ, 1.5σ, 2σ, 3σ, 5σ\} where σ = `returns.std()`.  

### Cell 9 — Visualization & Decision

Plotly dark-theme dashboard (2×2 subplot). Decision rule: minimize \|ACF(1)\| subject to n ≥ 5,000.

---

## Phase 2 — Triple Barrier Optimization &ensp; ✅ COMPLETE

> **Objective**: Find TB parameters that maximize label information content (uniqueness × entropy) while minimizing autocorrelation.  
> **Why**: Current config (PT=SL=1σ, h=30) produces labels that resolve in 1–2 bars → 70.9% persistence baseline.  
> **Input**: `df_bars_best` from Phase 1  
> **Output**: `df_labeled_best` (Polars DataFrame, 7,584 × 17), `best_tb_config` (dict)

**Cells**: 10 (OHLCV + volatility), 11 (TB grid sweep), 12 (rank + viz + export)  
**MLflow experiment**: `/Users/aaronng.dev1@gmail.com/BTCUSDT_Phase2_TripleBarrier`

### Results

| Metric | Target | Actual | Status |
|---|---|---|---|
| Label entropy | > 0.95 bits | **0.9998** | ✓ PASS |
| Average uniqueness | > 0.50 | 0.4856 | ✗ MISS |
| \|ACF(1)\| of labels | < 0.10 | **0.0413** | ✓ PASS |
| \|ACF(5)\| of labels | < 0.05 | **0.0034** | ✓ PASS |
| Median bars-until-touch | > 5 | 1.0 | ✗ MISS |
| Vertical barrier % | < 30% | **0.0%** | ✓ PASS |
| Class balance | ∈ [0.40, 0.60] | **0.4921** | ✓ PASS |

**Scorecard**: 5/7 targets met.

| Parameter | Winner |
|---|---|
| Config name | `pt0.5_sl0.5_h480_v20` |
| PT multiplier | 0.5σ |
| SL multiplier | 0.5σ |
| Max holding | 480 bars |
| Vol EWMA span | 20 |
| Score | 0.4655 (uniq × ent × (1−\|ACF₁\|)) |
| Labels | 7,584 (TP \~3,816 / SL \~3,768 / Vert=0) |

**Key insight**: Stride-1 event sampling produces heavy label overlap — median touch = 1 bar, avg uniqueness = 0.486. No PT/SL/hold combination can fix this structural property. **Phase 5 (sample weighting + sequential bootstrap) is mission-critical** to compensate for label non-independence.

### Hybrid Architecture: Polars ↔ NumPy

> **Design principle**: Keep data in Polars as long as the operations are columnar/declarative.  
> Drop to NumPy only for the sequential-scan TB inner loop and label-quality metrics.  
> Return results to Polars immediately for ranking/visualization.

### Cell 10 — OHLCV Preparation & Volatility Estimation &ensp; ☞ **POLARS**

Median intra-bar range is **0.29%** — barriers ≥ 2σ ($278+) needed for labels beyond intra-bar noise.

### Cell 11 — TB Parameter Grid Sweep &ensp; ☞ **NUMPY (wavefront)**

**Grid** (864 configs): PT × SL ∈ \{0.5,1,1.5,2,3,5\}, hold ∈ \{10,30,60,120,240,480\}, vol\_span ∈ \{20,50,100,200\}  
**Output**: `sweep_results` (295 configs evaluated), `tb_label_arrays` (top 20)

### Cell 12 — Rank, Visualize & Export &ensp; ☞ **POLARS + PLOTLY**

PT×SL score heatmap + touch-bar histograms. Export `df_labeled_best` and `best_tb_config`.

---

## Phase 3 — Feature Stationarity & Fracdiff &ensp; ✅ COMPLETE

> **Objective**: Make all features stationary via minimum fractional differentiation while preserving memory (correlation with original > 0.90).  
> **Why**: Price-scale features (SMA \~$70K, EMA \~$70K) have unit roots — learned thresholds drift across train/test.  
> **Input**: `df_bars_best` from Phase 1  
> **Output**: `fracdiff_map` (dict: 27 features → d\*), `df_features_stationary` (4,154 × 44)

**Cells**: 13 (indicators), 14 (ADF/KPSS audit), 15 (FFD search), 16 (viz + export)  
**MLflow experiment**: `/Users/aaronng.dev1@gmail.com/BTCUSDT_Phase3_Stationarity`

### Results

| Metric | Value |
|---|---|
| Features computed | 33 (schema-adaptive: skips absent source columns) |
| Features tested | 27 (6 exempt: binary + cyclical) |
| Stationary (d=0) | **14** — RSI, MACD (all 3), BB\_pct, all returns, vol\_ratio, trade\_intensity, body\_size, lower\_shadow |
| Non-stationary | **9** — all 7 MAs (SMA\*, EMA\*), BB\_upper, BB\_lower |
| Trend-stationary | **8** — BB\_std, BB\_width, ATR\_14, ATR\_pct, TR, vol\_sma\_10, vol\_sma\_20, upper\_shadow |
| FFD-transformed (d>0) | **9** features (mean d\*=0.117, range 0.05–0.15) |
| Mean corr at d\* | **0.9824** — near-perfect memory preservation |

**Key insight**: BTC volume bars are *almost* stationary — only a tiny fractional nudge (d=0.05–0.15) eliminates unit roots while preserving 95–99% of predictive memory.

> **⚠ Note**: Phase 4 superseded the FFD approach by adopting **stationary-by-construction** features (e.g., `sma_dist_5 = close/SMA - 1` instead of raw `sma_5`). All Phase 4 features have d\*=0 by design, making FFD unnecessary for the final feature set. Phase 3's FFD results remain valid as a diagnostic but are not used downstream.

**Diagnoses addressed**: #1 (non-stationary features → FFD applied) and #4 (no stationarity testing → full ADF/KPSS audit on all features).

---

## Phase 4 — Feature Engineering & MI Scoring &ensp; ✅ COMPLETE

> **Objective**: Expand the feature space beyond daily-equity defaults with microstructure, entropy, and structural-break features, all stationary-by-construction. Score via mutual information against optimized TB labels.  
> **AFML Reference**: Ch. 5 §5.4, Ch. 18 (microstructure), Ch. 19 (VPIN), Ch. 20 (entropy)  
> **Root Causes Addressed**: #1 (all features now stationary-by-construction), #6 (149 features across 3 tiers replace 39 flat defaults)  
> **Input**: `df_ticks` (52.5M × 5), `df_bars_best` (8,064 × 7), `df_labeled_best` (7,584 × 17)  
> **Output**: `df_features_all` (8,064 × 166), `df_mi_sweep` (132 scored features), `X` (7,385 × 132), `noise.p95`

**Cells**: 17 (plan), 18 (refactoring plan), 20 (4A microstructure), 21 (4B entropy/TA), 22 (4B-2 MI scoring)

### Data Flow

```
52.5M ticks ──[4A]──▶ 8,064 bars + 30 microstructure features (Tier 1)
                         │
                    ──[4B]──▶ + 18 entropy/structural (Tier 2) + 84 TA window sweep (Tier 3)
                         │
                         ▼
                    149 total features → drop >10% NaN → 132 scored
                         │
                    ──[4B-2]──▶ MI + noise floor + LightGBM AUC
                         │
                         ▼
                    7,385 usable rows × 132 features
```

### Results

| Metric | Value |
|---|---|
| Total features | 149 (30 Tier 1 + 18 Tier 2 + 84 Tier 3 + 17 base) |
| Scored after NaN filter | **132** (dropped `cs_spread_raw` — 99.97% NaN) |
| Usable rows | 7,385 / 7,584 (97.4% retained after NaN drop) |
| Features above MI noise floor | **0 / 132** |
| Noise floor (95th pct) | 0.014773 |
| Best MI score | 0.011274 (`macd_line_8_21_5`) |
| Best single-feature AUC | 0.5425 |

**Tier breakdown**:

| Tier | Count | Description |
|---|---|---|
| Tier 1 — Microstructure | 30 | VPIN, Kyle's Lambda, Amihud illiquidity, Roll/CS spreads, order imbalance (from raw ticks) |
| Tier 2 — Entropy/Structural | 18 | Shannon entropy, Lempel-Ziv complexity, CUSUM statistic, rolling ADF, martingale difference |
| Tier 3 — Traditional TA (swept) | 84 | SMA/EMA distance, RSI, MACD, Bollinger, ATR at multiple window sizes — all stationary-by-construction |

**Top 10 features by MI**:

| Rank | Feature | Tier | Family | MI | AUC |
|---|---|---|---|---|---|
| 1 | `macd_line_8_21_5` | Tier 3 | MACD | 0.01127 | 0.5276 |
| 2 | `macd_line_12_26_9` | Tier 3 | MACD | 0.01080 | 0.5335 |
| 3 | `return_20` | Tier 3 | Returns | 0.01043 | 0.5261 |
| 4 | `macd_signal_8_21_5` | Tier 3 | MACD | 0.01034 | 0.5253 |
| 5 | `return_10` | Tier 3 | Returns | 0.00998 | 0.5290 |
| 6 | `macd_signal_12_26_9` | Tier 3 | MACD | 0.00995 | 0.5279 |
| 7 | `sma_dist_50` | Tier 3 | SMA Distance | 0.00984 | 0.5254 |
| 8 | `rsi_50` | Tier 3 | RSI | 0.00979 | 0.5312 |
| 9 | `ema_dist_50` | Tier 3 | EMA Distance | 0.00943 | 0.5234 |
| 10 | `bb_pct_50` | Tier 3 | Bollinger | 0.00939 | 0.5271 |

**⚠ Critical finding**: 0/132 features exceed the 95th-percentile permutation noise floor (0.0148). The best MI (0.0113) is only 0.76× the noise threshold. Single-feature AUCs cluster near 0.50–0.54. This suggests the optimized TB labels may not contain learnable signal at the feature level — **sample weighting (Phase 5) is essential** to reduce effective noise from overlapping labels.

### Design Decisions

* **Stationary-by-construction**: Phase 4 features use relative/normalized forms (e.g., `sma_dist_5 = close/SMA - 1`) instead of raw price-scale values. All 132 features are stationary at d\*=0 — no FFD needed.
* **3-tier architecture**: Microstructure features (from raw 52.5M ticks) add market microstructure signal unavailable in OHLCV-only indicators. Entropy/structural features capture regime change.
* **Configurable via frozen dataclasses**: `TechnicalConfig`, `MicrostructureConfig`, `EntropyConfig`, `StructuralConfig` — reproducible and testable.

### Cell 17 — Phase 4 Implementation Plan (markdown)

Detailed plan covering 3-tier feature architecture, MI scoring methodology, and empirical motivation from prototype results.

### Cell 18 — Refactoring Plan (markdown)

Subpackage refactoring plan — current vs target structure, migration phases, backward-compat strategy.

### Cell 20 — Phase 4A: Microstructure Features &ensp; ☞ **POLARS + NUMPY**

Calls `compute_microstructure_features(df_ticks, df_bars_best)` from `btcusdt_utils.features`.  
Aggregates 52.5M ticks into bar-level features: VPIN, Kyle's Lambda, Amihud, Roll/Corwin-Schultz spreads, order flow imbalance.  
**Output**: `df_bars_micro` (8,064 × 37+), `micro_feature_names` (30 features)

### Cell 21 — Phase 4B: Entropy, Structural & TA Sweep &ensp; ☞ **NUMPY + POLARS**

Computes Tier 2 (entropy + structural break) and Tier 3 (window-swept TA) features:  
&emsp; Entropy: `compute_entropy_features(log_ret)` — Shannon, Lempel-Ziv at multiple windows  
&emsp; Structural: `compute_structural_features(log_ret)` — CUSUM, rolling ADF, martingale  
&emsp; TA: `compute_technical_indicators(df_bars_best, TechnicalConfig(...))` — stationary-by-construction  
Joins all tiers into `df_features_all`.  
**Output**: `df_features_all` (8,064 × 166), `tier2_feature_names`, `tier3_feature_names`

### Cell 22 — Phase 4B-2: MI Scoring & Feature Evaluation

Scores all 132 features via:  
&emsp; `compute_mi_scores(X, y)` — mutual information with TB labels  
&emsp; `compute_noise_floor(X, y)` — 100-permutation noise threshold  
&emsp; `compute_univariate_auc(X, y)` — single-feature LightGBM AUC  
**Output**: `df_mi_sweep` (132 × 7), `X` (7,385 × 132), `noise`

---

## Phase 5 — Sample Weighting &ensp; ⬜ TODO

> **Objective**: Compute per-sample uniqueness weights so overlapping TB labels don't dominate training.  
> **Why**: With 30+ bar holding period at 1-bar sampling, each sample shares price data with \~30 neighbors. Effective N \~800.  
> **⚠ MISSION-CRITICAL**: Phase 2 gap analysis confirmed avg\_uniqueness = 0.486 (below 0.50 target). Phase 4 confirmed 0/132 features above noise floor — label overlap likely the primary cause.  
> **Input**: optimized TB labels + event boundaries from Phase 2  
> **Output**: `sample_weights` (np.ndarray), `avg_uniqueness` (float)

### Concurrency Matrix & Uniqueness Weights

1. Build concurrency matrix: `C[t] = count of active labels at bar t`  
2. Per-sample uniqueness: `u_i = mean(1 / C[t])` for `t ∈ [start_i, end_i]`  
3. Sample weight: `w_i = u_i` (optionally scaled by `|return_i|` for return attribution)  

### Sequential Bootstrap Validation

1. Sequential bootstrap (AFML §4.5.2): draw samples preferring high-uniqueness candidates  
2. Compare: weighted LightGBM vs unweighted → measure AUC delta on holdout  

---

## Phase 6 — Purged K-Fold CV &ensp; ⬜ TODO

> **Objective**: Implement cross-validation that respects TB label overlap, preventing information leakage at fold boundaries.  
> **Why**: Chronological split without purging leaks the last `max_holding` training labels into validation.  
> **Input**: feature matrix + labels from Phases 2–4, sample weights from Phase 5  
> **Output**: `PurgedKFold` class, CV results

Sweep `embargo_pct ∈ {0.005, 0.01, 0.02}`. Compare CV variance: purged vs standard chronological split.

---

## Phase 7 — Retrain & Compare &ensp; ⬜ TODO

> **Objective**: Apply all optimized parameters, retrain the same 5 models, and compare against the archived 0.519 AUC.  
> **Input**: all outputs from Phases 1–6  
> **Output**: final model comparison table, verdict

Same 5 models: LightGBM, XGBoost, RandomForest, LogisticRegression, DecisionTree.  
Optuna 20 trials per model. Use `sample_weight` + PurgedKFold.

**Success criteria**:  
- ROC-AUC > 0.55 (meaningful signal above noise)  
- Beat "Last TB Label" persistence baseline (0.708)  
- Label ACF(1) < 0.10  
- Average uniqueness > 0.5  

If success criteria met → register to Unity Catalog: `smart-trading-dev.03_gold.btcusdt_direction_v2`  
If not → document what improved, what didn't, and propose next iteration.

In [0]:
# PHASE 0A — Install Dependencies
# ============================================================
# Install ALL required packages (environment-agnostic).
# fracdiff: no compatible wheel — FFD implemented manually in 0B
# ============================================================

%pip install polars lightgbm mlflow statsmodels plotly scikit-learn --quiet

In [0]:
# Restart kernel to pick up newly installed packages
%restart_python

In [0]:
# PHASE 0B — Constants, Imports & Utility Package
# ============================================================
# Re-establishes all state after kernel restart.
# Loads reusable helpers from btcusdt_utils package.
# ============================================================

from __future__ import annotations

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# ── Standard library ──────────────────────────────────────────
from math import pi as PI

# ── Third-party ───────────────────────────────────────────────
import numpy as np
import polars as pl
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from pyspark.sql import functions as F, Window
from scipy import stats as sp_stats
import mlflow

print("✓ Imports OK")

# ── Project utilities ─────────────────────────────────────────
import sys
sys.path.insert(0, "/Workspace/Users/aaronng.dev1@gmail.com")

from btcusdt_utils import (  # noqa: E402
    # Constants
    CATALOG, BRONZE, SILVER, GOLD, RAW_AGGTRADES, RAW_AGGTRADES_PATH,
    # Viz theme
    _BG, _BG_CARD, _TEXT, _TEXT_MUT, _GRID,
    _RED, _GREEN, _BLUE, _ORANGE, _PURPLE, _CYAN,
    _FONT, DARK_LAYOUT, dark_fig,
    # Quality metrics
    BarQuality, LabelQuality,
    compute_bar_quality, compute_label_quality,
    label_entropy, label_avg_uniqueness,
    # Stationarity & FFD
    adf_test, kpss_test, ffd, find_min_ffd,
    # Bar construction
    build_time_bars, build_threshold_bars, cusum_filter,
    # Labeling
    apply_triple_barrier,
    # Features (subpackage)
    TechnicalConfig, MicrostructureConfig, EntropyConfig, StructuralConfig,
    compute_technical_indicators, compute_microstructure_features,
    compute_entropy_features, compute_structural_features,
    # Scoring (subpackage)
    ScoringConfig, NoiseFloor,
    compute_mi_scores, compute_noise_floor, compute_univariate_auc,
)

print(f"✓ Source table: {RAW_AGGTRADES_PATH}")
print("✓ Plotly dark theme configured")
print("✓ All helpers loaded from btcusdt_utils v0.3.0")
print(f"\n{'=' * 70}")
print("✓ Phase 0B complete — all constants, imports & helpers ready")
print(f"{'=' * 70}")

In [0]:
# PHASE 0C — Load Raw Tick Data & Verify
# ============================================================
# Loads the full 52.5M tick dataset from Bronze layer.
# Extracts core statistics, deduplicates, and collects to
# Polars for fast local bar construction in Phase 1.
# ============================================================

print("=" * 70)
print("PHASE 0C: LOAD RAW TICK DATA")
print("=" * 70)

# ── 1. Load from Delta ────────────────────────────────────────
df_raw_spark = spark.table(RAW_AGGTRADES_PATH)

# ── 2. Core stats (single distributed pass) ───────────────────
print("▸ Computing core statistics (single Spark pass)...")

core_stats_df = df_raw_spark.agg(
    F.count("*").alias("total_rows"),
    F.countDistinct("agg_trade_id").alias("distinct_ids"),
    F.min("transact_timestamp").alias("min_ts"),
    F.max("transact_timestamp").alias("max_ts"),
    F.min("price").alias("min_price"),
    F.max("price").alias("max_price"),
    F.avg("price").alias("avg_price"),
    F.stddev("price").alias("stddev_price"),
    F.avg("quantity").alias("avg_qty"),
    F.avg((F.col("price") * F.col("quantity")).cast("double")).alias("avg_dollar"),
)

core = core_stats_df.first()

# ── 3. Extract scalars ────────────────────────────────────────
TOTAL_ROWS   = core["total_rows"]
DISTINCT_IDS = core["distinct_ids"]
MIN_TS       = core["min_ts"]
MAX_TS       = core["max_ts"]
MIN_PRICE    = float(core["min_price"])
MAX_PRICE    = float(core["max_price"])
AVG_PRICE    = float(core["avg_price"])
AVG_QTY      = float(core["avg_qty"])
AVG_DOLLAR   = float(core["avg_dollar"])
TIME_SPAN    = (MAX_TS - MIN_TS).total_seconds() / 86400

# ── 4. Summary ────────────────────────────────────────────────
print(f"\n{'─' * 70}")
print("DATA SUMMARY")
print(f"{'─' * 70}")
print(f"  Total rows:       {TOTAL_ROWS:>14,}")
print(f"  Distinct IDs:     {DISTINCT_IDS:>14,}")
print(f"  Duplicates:       {TOTAL_ROWS - DISTINCT_IDS:>14,}")
print(f"  Time range:       {MIN_TS} → {MAX_TS}")
print(f"  Time span:        {TIME_SPAN:>14.2f} days")
print(f"  Price range:      ${MIN_PRICE:>11,.2f} – ${MAX_PRICE:,.2f}")
print(f"  Avg price:        ${AVG_PRICE:>11,.2f}")
print(f"  Avg quantity:     {AVG_QTY:>14.6f} BTC")
print(f"  Avg dollar/trade: ${AVG_DOLLAR:>11,.2f}")
print(f"  Avg trades/min:   {TOTAL_ROWS / (TIME_SPAN * 1440):>14,.0f}")

# ── 5. Deduplicate + select + cast (Spark) ────────────────────
print(f"\n▸ Deduplicating and preparing columns...")
df_prepped = (
    df_raw_spark
    .dropDuplicates(["agg_trade_id"])
    .select(
        "transact_timestamp",
        F.col("price").cast("double").alias("price_d"),
        F.col("quantity").cast("double").alias("qty_d"),
        (F.col("price") * F.col("quantity")).cast("double").alias("dollar_d"),
        F.col("is_buyer_maker"),
    )
    .orderBy("transact_timestamp")
)

RAW_COUNT = df_prepped.count()
print(f"  After dedup: {RAW_COUNT:,} rows")

# ── 6. Collect to Polars via Arrow (zero-copy) ────────────────
# Spark → Arrow → Polars (near-zero overhead, no Pandas detour)
print(f"▸ Collecting {RAW_COUNT:,} rows to Polars via toArrow()...")
df_ticks = pl.from_arrow(df_prepped.toArrow())

print(f"  Polars shape: {df_ticks.shape}")
print(f"  Memory: ~{df_ticks.estimated_size('mb'):.0f} MB")
print(f"  Columns: {df_ticks.columns}")
print(f"  dtypes:  {[str(d) for d in df_ticks.dtypes]}")

# Quick sanity check
assert df_ticks.shape[0] > 50_000_000, f"Expected >50M rows, got {df_ticks.shape[0]:,}"
assert df_ticks["price_d"].null_count() == 0, "Unexpected nulls in price_d"

print(f"\n{'=' * 70}")
print(f"✓ Phase 0C complete — {df_ticks.shape[0]:,} clean ticks in Polars")
print(f"{'=' * 70}")

In [0]:
# PHASE 1A — Compute Bar Candidates
# ============================================================
# Builds 15 bar configurations from df_ticks:
#   - 4 time bars: {1, 5, 15, 30} min
#   - 4 volume bars: {1×, 2×, 5×, 10×} median per-minute volume
#   - 4 dollar bars: {1×, 2×, 5×, 10×} median per-minute dollar
#   - 3 tick bars: {1×, 2×, 5×} median per-minute tick count
#
# Uses MEDIAN (not mean) for threshold calibration — robust to
# the Feb 4-6 drawdown regime event.
# ============================================================

import time as _time

print("=" * 70)
print("PHASE 1A: COMPUTE BAR CANDIDATES")
print("=" * 70)

# ═══════════════════════════════════════════════════════════════
# 1. COMPUTE MEDIAN PER-MINUTE STATISTICS
# ═══════════════════════════════════════════════════════════════
# Group ticks by minute, compute per-minute aggregates, then take median.
# This is more robust than using averages from Phase 0C.

print("\n▸ Computing median per-minute statistics...")

per_minute_stats = (
    df_ticks
    .sort("transact_timestamp")
    .group_by_dynamic("transact_timestamp", every="1m")
    .agg(
        pl.col("qty_d").sum().alias("minute_volume"),
        pl.col("dollar_d").sum().alias("minute_dollar"),
        pl.len().alias("minute_ticks"),
    )
)

MEDIAN_VOLUME_PER_MIN = float(per_minute_stats["minute_volume"].median())
MEDIAN_DOLLAR_PER_MIN = float(per_minute_stats["minute_dollar"].median())
MEDIAN_TICKS_PER_MIN  = float(per_minute_stats["minute_ticks"].median())

print(f"  Median volume/min:  {MEDIAN_VOLUME_PER_MIN:>12,.2f} BTC")
print(f"  Median dollar/min:  ${MEDIAN_DOLLAR_PER_MIN:>12,.0f}")
print(f"  Median ticks/min:   {MEDIAN_TICKS_PER_MIN:>12,.0f}")

# ═══════════════════════════════════════════════════════════════
# 2. DEFINE BAR CONFIGURATIONS
# ═══════════════════════════════════════════════════════════════

TIME_INTERVALS = [1, 5, 15, 30]  # minutes
THRESHOLD_MULTIPLIERS = [1, 2, 5, 10]
TICK_MULTIPLIERS = [1, 2, 5]  # Fewer for tick bars per spec

bar_configs: list[dict] = []

# Time bars
for interval in TIME_INTERVALS:
    bar_configs.append({
        "name": f"time_{interval}m",
        "type": "time",
        "interval": interval,
    })

# Volume bars
for mult in THRESHOLD_MULTIPLIERS:
    bar_configs.append({
        "name": f"volume_{mult}x",
        "type": "volume",
        "threshold": MEDIAN_VOLUME_PER_MIN * mult,
        "column": "qty_d",
    })

# Dollar bars
for mult in THRESHOLD_MULTIPLIERS:
    bar_configs.append({
        "name": f"dollar_{mult}x",
        "type": "dollar",
        "threshold": MEDIAN_DOLLAR_PER_MIN * mult,
        "column": "dollar_d",
    })

# Tick bars
for mult in TICK_MULTIPLIERS:
    bar_configs.append({
        "name": f"tick_{mult}x",
        "type": "tick",
        "threshold": MEDIAN_TICKS_PER_MIN * mult,
        "column": "tick",
    })

print(f"\n▸ Defined {len(bar_configs)} bar configurations")

# ═══════════════════════════════════════════════════════════════
# 3. BUILD ALL BAR CANDIDATES
# ═══════════════════════════════════════════════════════════════

bar_results: dict[str, dict] = {}

print(f"\n{'─' * 70}")
print("BUILDING BAR CANDIDATES")
print(f"{'─' * 70}")

for cfg in bar_configs:
    name = cfg["name"]
    start = _time.perf_counter()
    
    try:
        if cfg["type"] == "time":
            df_bars = build_time_bars(df_ticks, cfg["interval"])
        else:
            df_bars = build_threshold_bars(
                df_ticks,
                threshold=cfg["threshold"],
                column=cfg["column"],
                bar_type=cfg["type"],
            )
        
        # Compute log returns
        close = df_bars["close"].to_numpy()
        returns = np.log(close[1:] / close[:-1])
        returns = returns[np.isfinite(returns)]
        
        elapsed = _time.perf_counter() - start
        
        bar_results[name] = {
            "config": cfg,
            "df_bars": df_bars,
            "returns": returns,
            "n_bars": len(df_bars),
            "n_returns": len(returns),
            "elapsed_s": elapsed,
        }
        
        status = "✓" if len(returns) >= 5000 else "⚠ <5K"
        print(f"  {status} {name:<15} → {len(df_bars):>8,} bars  ({elapsed:.2f}s)")
        
    except ValueError as e:
        elapsed = _time.perf_counter() - start
        print(f"  ✗ {name:<15} → FAILED: {e}")
        bar_results[name] = {
            "config": cfg,
            "df_bars": None,
            "returns": None,
            "n_bars": 0,
            "n_returns": 0,
            "elapsed_s": elapsed,
            "error": str(e),
        }

# ═══════════════════════════════════════════════════════════════
# 4. SUMMARY
# ═══════════════════════════════════════════════════════════════

valid_configs = [k for k, v in bar_results.items() if v["n_returns"] >= 30]
ml_ready_configs = [k for k, v in bar_results.items() if v["n_returns"] >= 5000]

print(f"\n{'─' * 70}")
print("SUMMARY")
print(f"{'─' * 70}")
print(f"  Total configurations:    {len(bar_configs)}")
print(f"  Valid (≥30 returns):     {len(valid_configs)}")
print(f"  ML-ready (≥5K returns):  {len(ml_ready_configs)}")
print(f"\n{'=' * 70}")
print("✓ Phase 1A complete — bar candidates ready for quality evaluation")
print(f"{'=' * 70}")

In [0]:
# PHASE 1B — Evaluate Bar Quality & Log to MLflow
# ============================================================
# For each bar candidate:
#   1. Compute IID-ness metrics via compute_bar_quality()
#   2. Log to MLflow experiment for traceability
#   3. Rank by |ACF(1)| subject to n ≥ 5,000
# ============================================================

print("=" * 70)
print("PHASE 1B: EVALUATE BAR QUALITY")
print("=" * 70)

# ═══════════════════════════════════════════════════════════════
# 1. SETUP MLFLOW EXPERIMENT
# ═══════════════════════════════════════════════════════════════

EXPERIMENT_NAME = "/Users/aaronng.dev1@gmail.com/BTCUSDT_Phase1_BarSampling"

mlflow.set_experiment(EXPERIMENT_NAME)
print(f"\n▸ MLflow experiment: {EXPERIMENT_NAME}")

# ═══════════════════════════════════════════════════════════════
# 2. EVALUATE BAR QUALITY FOR ALL VALID CONFIGS
# ═══════════════════════════════════════════════════════════════

MIN_RETURNS_FOR_QUALITY = 30
MIN_RETURNS_FOR_ML = 5000

quality_results: list[BarQuality] = []

print(f"\n{'─' * 70}")
print("COMPUTING QUALITY METRICS")
print(f"{'─' * 70}")

with mlflow.start_run(run_name="phase1_bar_sweep") as parent_run:
    # Log parent-level params
    mlflow.log_params({
        "n_ticks": len(df_ticks),
        "median_volume_per_min": MEDIAN_VOLUME_PER_MIN,
        "median_dollar_per_min": MEDIAN_DOLLAR_PER_MIN,
        "median_ticks_per_min": MEDIAN_TICKS_PER_MIN,
        "min_returns_for_ml": MIN_RETURNS_FOR_ML,
    })
    
    for name, result in bar_results.items():
        returns = result.get("returns")
        
        if returns is None or len(returns) < MIN_RETURNS_FOR_QUALITY:
            print(f"  ✗ {name:<15} — skipped (insufficient returns)")
            continue
        
        # Compute quality metrics
        bq = compute_bar_quality(returns, name)
        quality_results.append(bq)
        
        # Determine status
        ml_ready = bq.n_bars >= MIN_RETURNS_FOR_ML
        acf_ok = abs(bq.acf_1) < 0.05
        status = "✓" if ml_ready and acf_ok else ("⚠" if ml_ready else "○")
        
        print(
            f"  {status} {name:<15} │ n={bq.n_bars:>7,} │ "
            f"ACF1={bq.acf_1:>+.4f} │ LB_Q10={bq.lb_stat:>8.1f} │ "
            f"JB={bq.jb_stat:>10.1f}"
        )
        
        # Log as nested run
        with mlflow.start_run(run_name=name, nested=True):
            cfg = result["config"]
            mlflow.log_params({
                "bar_type": cfg["type"],
                "bar_name": name,
                "threshold": cfg.get("threshold", cfg.get("interval")),
            })
            mlflow.log_metrics({
                "n_bars": bq.n_bars,
                "acf_1": bq.acf_1,
                "abs_acf_1": abs(bq.acf_1),
                "lb_stat_10": bq.lb_stat,
                "lb_pval_10": bq.lb_pval,
                "jb_stat": bq.jb_stat,
                "jb_pval": bq.jb_pval,
                "ret_mean": bq.ret_mean,
                "ret_std": bq.ret_std,
                "cv": bq.cv,
                "ml_ready": float(ml_ready),
            })

# ═══════════════════════════════════════════════════════════════
# 3. RANK CONFIGURATIONS
# ═══════════════════════════════════════════════════════════════
# Decision rule: minimize |ACF(1)| subject to n ≥ 5,000

print(f"\n{'─' * 70}")
print("RANKING (sorted by |ACF(1)|, filtered to n ≥ 5,000)")
print(f"{'─' * 70}")

# Filter to ML-ready, sort by |ACF(1)|
ml_ready_quality = sorted(
    (bq for bq in quality_results if bq.n_bars >= MIN_RETURNS_FOR_ML),
    key=lambda bq: abs(bq.acf_1),
)

print(f"\n{'Rank':<5} {'Config':<15} {'n_bars':>10} {'|ACF(1)|':>10} {'LB p-val':>10} {'Status':<10}")
print("─" * 65)

for rank, bq in enumerate(ml_ready_quality, start=1):
    lb_status = "IID" if bq.lb_pval > 0.05 else "ACF"
    acf_status = "✓" if abs(bq.acf_1) < 0.05 else "⚠"
    print(
        f"{rank:<5} {bq.name:<15} {bq.n_bars:>10,} {abs(bq.acf_1):>10.5f} "
        f"{bq.lb_pval:>10.4f} {acf_status} {lb_status}"
    )

# Identify best config
if ml_ready_quality:
    best_bar_config_name = ml_ready_quality[0].name
    best_bar_quality = ml_ready_quality[0]
    print(f"\n★ Best bar type (pre-CUSUM): {best_bar_config_name}")
    print(f"  |ACF(1)| = {abs(best_bar_quality.acf_1):.5f}, n = {best_bar_quality.n_bars:,}")
else:
    best_bar_config_name = None
    best_bar_quality = None
    print("\n⚠ No ML-ready configurations found!")

print(f"\n{'=' * 70}")
print("✓ Phase 1B complete — quality metrics logged to MLflow")
print(f"{'=' * 70}")

In [0]:
# PHASE 1C — CUSUM Filter Overlay
# ============================================================
# Apply CUSUM event filter (AFML Ch. 2.5.2) on the best bar
# type from Phase 1B.
#
# Sweep thresholds: h ∈ {1σ, 1.5σ, 2σ, 3σ, 5σ}
# Adopt if IID-ness improves without dropping below 5K samples.
# ============================================================

print("=" * 70)
print("PHASE 1C: CUSUM FILTER OVERLAY")
print("=" * 70)

if best_bar_config_name is None:
    print("\n⚠ No best bar config found — skipping CUSUM overlay")
else:
    # Get the best bar's data
    best_result = bar_results[best_bar_config_name]
    df_best_bars = best_result["df_bars"]
    base_returns = best_result["returns"]
    base_std = float(np.std(base_returns))
    
    print(f"\n▸ Base config: {best_bar_config_name}")
    print(f"  Base σ (return std): {base_std:.6f}")
    print(f"  Base n: {len(base_returns):,}")
    print(f"  Base |ACF(1)|: {abs(best_bar_quality.acf_1):.5f}")
    
    # ═════════════════════════════════════════════════════════════
    # SWEEP CUSUM THRESHOLDS
    # ═════════════════════════════════════════════════════════════
    
    CUSUM_MULTIPLIERS = [1.0, 1.5, 2.0, 3.0, 5.0]
    
    cusum_results: list[dict] = []
    
    # Add baseline (no CUSUM) for comparison
    cusum_results.append({
        "name": f"{best_bar_config_name}_no_cusum",
        "h_mult": 0.0,
        "h": 0.0,
        "quality": best_bar_quality,
        "n_events": len(base_returns),
        "filter_ratio": 1.0,
    })
    
    print(f"\n{'─' * 70}")
    print("CUSUM SWEEP RESULTS")
    print(f"{'─' * 70}")
    print(f"{'Config':<25} {'h':>10} {'n_events':>10} {'|ACF(1)|':>10} {'LB p-val':>10} {'ΔACF':>8}")
    print("─" * 80)
    
    # Baseline row
    print(
        f"{best_bar_config_name + ' (base)':<25} {'-':>10} {len(base_returns):>10,} "
        f"{abs(best_bar_quality.acf_1):>10.5f} {best_bar_quality.lb_pval:>10.4f} {'-':>8}"
    )
    
    for mult in CUSUM_MULTIPLIERS:
        h = base_std * mult
        name = f"{best_bar_config_name}_cusum_{mult}σ"
        
        # Apply CUSUM filter
        event_indices = cusum_filter(base_returns, h)
        
        if len(event_indices) < MIN_RETURNS_FOR_QUALITY:
            print(f"{name:<25} {h:>10.6f} {len(event_indices):>10,} {'--- too few ---':>30}")
            continue
        
        # Subsample bars at CUSUM events
        # Note: event_indices are indices into returns array, which is 1 shorter than bars
        # We take bars at event_indices + 1 (the bar that completed the return)
        cusum_bar_indices = event_indices + 1
        cusum_bar_indices = cusum_bar_indices[cusum_bar_indices < len(df_best_bars)]
        
        if len(cusum_bar_indices) < MIN_RETURNS_FOR_QUALITY:
            print(f"{name:<25} {h:>10.6f} {len(cusum_bar_indices):>10,} {'--- too few ---':>30}")
            continue
        
        # Get close prices at CUSUM events and compute returns
        close_arr = df_best_bars["close"].to_numpy()
        cusum_close = close_arr[cusum_bar_indices]
        cusum_returns = np.log(cusum_close[1:] / cusum_close[:-1])
        cusum_returns = cusum_returns[np.isfinite(cusum_returns)]
        
        if len(cusum_returns) < MIN_RETURNS_FOR_QUALITY:
            print(f"{name:<25} {h:>10.6f} {len(cusum_returns):>10,} {'--- too few returns ---':>30}")
            continue
        
        # Compute quality
        try:
            cusum_quality = compute_bar_quality(cusum_returns, name)
            delta_acf = abs(cusum_quality.acf_1) - abs(best_bar_quality.acf_1)
            
            cusum_results.append({
                "name": name,
                "h_mult": mult,
                "h": h,
                "quality": cusum_quality,
                "n_events": cusum_quality.n_bars,
                "filter_ratio": cusum_quality.n_bars / len(base_returns),
                "cusum_bar_indices": cusum_bar_indices,
            })
            
            improved = "↓" if delta_acf < 0 else ("↑" if delta_acf > 0 else "=")
            print(
                f"{name:<25} {h:>10.6f} {cusum_quality.n_bars:>10,} "
                f"{abs(cusum_quality.acf_1):>10.5f} {cusum_quality.lb_pval:>10.4f} "
                f"{delta_acf:>+7.5f} {improved}"
            )
        except Exception as e:
            print(f"{name:<25} {h:>10.6f} ERROR: {e}")
    
    # ═════════════════════════════════════════════════════════════
    # SELECT BEST CONFIGURATION
    # ═════════════════════════════════════════════════════════════
    # Decision: minimize |ACF(1)| subject to n ≥ 5,000
    
    ml_ready_cusum = [
        r for r in cusum_results 
        if r["quality"].n_bars >= MIN_RETURNS_FOR_ML
    ]
    
    if ml_ready_cusum:
        best_cusum = min(ml_ready_cusum, key=lambda r: abs(r["quality"].acf_1))
        
        print(f"\n{'─' * 70}")
        print("DECISION")
        print(f"{'─' * 70}")
        
        # Check if CUSUM improves over baseline
        if best_cusum["h_mult"] > 0 and abs(best_cusum["quality"].acf_1) < abs(best_bar_quality.acf_1):
            print(f"★ CUSUM ADOPTED: {best_cusum['name']}")
            print(f"  h = {best_cusum['h_mult']}σ = {best_cusum['h']:.6f}")
            print(f"  n = {best_cusum['quality'].n_bars:,} (filter ratio: {best_cusum['filter_ratio']:.1%})")
            print(f"  |ACF(1)| = {abs(best_cusum['quality'].acf_1):.5f} (was {abs(best_bar_quality.acf_1):.5f})")
            
            USE_CUSUM = True
            CUSUM_H_MULT = best_cusum["h_mult"]
            CUSUM_H = best_cusum["h"]
            final_bar_config_name = best_cusum["name"]
            final_bar_quality = best_cusum["quality"]
            final_bar_indices = best_cusum.get("cusum_bar_indices")
        else:
            print(f"★ NO CUSUM: baseline {best_bar_config_name} is optimal")
            print(f"  (CUSUM did not improve |ACF(1)| while maintaining n ≥ 5,000)")
            
            USE_CUSUM = False
            CUSUM_H_MULT = None
            CUSUM_H = None
            final_bar_config_name = best_bar_config_name
            final_bar_quality = best_bar_quality
            final_bar_indices = None
    else:
        print("\n⚠ No configurations meet ML threshold after CUSUM filtering")
        USE_CUSUM = False
        final_bar_config_name = best_bar_config_name
        final_bar_quality = best_bar_quality
        final_bar_indices = None

print(f"\n{'=' * 70}")
print("✓ Phase 1C complete — CUSUM overlay evaluated")
print(f"{'=' * 70}")

In [0]:
# PHASE 1D — Visualization & Final Decision
# ============================================================
# 2×2 Plotly dark-theme dashboard:
#   (1,1) Bar chart: |ACF(1)| by configuration
#   (1,2) Bar chart: Ljung-Box Q-stat by configuration
#   (2,1) Scatter: sample count vs |ACF(1)| (Pareto frontier)
#   (2,2) Return distribution overlay (top 3 candidates)
#
# Exports: df_bars_best, best_bar_config (dict) for Phase 2
# ============================================================

print("=" * 70)
print("PHASE 1D: VISUALIZATION & FINAL DECISION")
print("=" * 70)

# ═══════════════════════════════════════════════════════════════
# 1. PREPARE DATA FOR VISUALIZATION
# ═══════════════════════════════════════════════════════════════

# Sort by |ACF(1)| for consistent ordering
all_quality_sorted = sorted(quality_results, key=lambda bq: abs(bq.acf_1))

names = [bq.name for bq in all_quality_sorted]
acf_vals = [abs(bq.acf_1) for bq in all_quality_sorted]
lb_stats = [bq.lb_stat for bq in all_quality_sorted]
n_bars_list = [bq.n_bars for bq in all_quality_sorted]

# Color by bar type
bar_type_colors = {
    "time": _BLUE,
    "volume": _GREEN,
    "dollar": _ORANGE,
    "tick": _PURPLE,
}

colors = [
    bar_type_colors.get(bar_results[bq.name]["config"]["type"], _TEXT)
    for bq in all_quality_sorted
]

# Highlight ML-ready configs
ml_ready_mask = [bq.n_bars >= MIN_RETURNS_FOR_ML for bq in all_quality_sorted]

# ═══════════════════════════════════════════════════════════════
# 2. BUILD 2×2 DASHBOARD
# ═══════════════════════════════════════════════════════════════

fig = dark_fig(
    rows=2, cols=2,
    subplot_titles=(
        "|ACF(1)| by Configuration",
        "Ljung-Box Q₁₀ by Configuration",
        "Sample Count vs |ACF(1)| (Pareto Frontier)",
        "Return Distribution (Top 3 Candidates)",
    ),
    vertical_spacing=0.12,
    horizontal_spacing=0.08,
)

# (1,1) ACF(1) bar chart
fig.add_trace(
    go.Bar(
        x=names,
        y=acf_vals,
        marker_color=colors,
        name="|ACF(1)|",
        showlegend=False,
    ),
    row=1, col=1,
)
fig.add_hline(
    y=0.05, line_dash="dash", line_color=_RED,
    annotation_text="Target: 0.05", row=1, col=1,
)

# (1,2) Ljung-Box Q-stat bar chart
fig.add_trace(
    go.Bar(
        x=names,
        y=lb_stats,
        marker_color=colors,
        name="LB Q₁₀",
        showlegend=False,
    ),
    row=1, col=2,
)

# (2,1) Pareto frontier scatter
for bar_type, color in bar_type_colors.items():
    mask = [bar_results[bq.name]["config"]["type"] == bar_type for bq in all_quality_sorted]
    if any(mask):
        fig.add_trace(
            go.Scatter(
                x=[n for n, m in zip(n_bars_list, mask) if m],
                y=[a for a, m in zip(acf_vals, mask) if m],
                mode="markers+text",
                marker=dict(size=12, color=color),
                text=[n for n, m in zip(names, mask) if m],
                textposition="top center",
                textfont=dict(size=9, color=_TEXT_MUT),
                name=bar_type.capitalize(),
                legendgroup=bar_type,
            ),
            row=2, col=1,
        )

# Add 5K threshold line
fig.add_vline(
    x=5000, line_dash="dash", line_color=_RED,
    annotation_text="ML threshold: 5K", row=2, col=1,
)
fig.add_hline(
    y=0.05, line_dash="dash", line_color=_CYAN,
    annotation_text="|ACF(1)| target", row=2, col=1,
)

# (2,2) Return distribution overlay for top 3 ML-ready configs
top_3 = [bq for bq in all_quality_sorted if bq.n_bars >= MIN_RETURNS_FOR_ML][:3]
dist_colors = [_BLUE, _GREEN, _ORANGE]

for bq, dist_color in zip(top_3, dist_colors):
    returns = bar_results[bq.name]["returns"]
    fig.add_trace(
        go.Histogram(
            x=returns,
            nbinsx=100,
            opacity=0.6,
            marker_color=dist_color,
            name=bq.name,
            legendgroup="dist",
        ),
        row=2, col=2,
    )

# Update layout
fig.update_layout(
    height=800,
    title=dict(
        text="Phase 1: Bar Sampling Strategy Comparison",
        font=dict(size=18),
    ),
    showlegend=True,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.15,
        xanchor="center",
        x=0.5,
    ),
)

# Axis labels
fig.update_xaxes(title_text="Configuration", row=1, col=1, tickangle=45)
fig.update_xaxes(title_text="Configuration", row=1, col=2, tickangle=45)
fig.update_xaxes(title_text="Sample Count (n)", row=2, col=1, type="log")
fig.update_xaxes(title_text="Log Return", row=2, col=2)

fig.update_yaxes(title_text="|ACF(1)|", row=1, col=1)
fig.update_yaxes(title_text="Q-statistic", row=1, col=2)
fig.update_yaxes(title_text="|ACF(1)|", row=2, col=1)
fig.update_yaxes(title_text="Count", row=2, col=2)

fig.show()

# ═══════════════════════════════════════════════════════════════
# 3. EXPORT FINAL OUTPUTS FOR PHASE 2
# ═══════════════════════════════════════════════════════════════

print(f"\n{'─' * 70}")
print("FINAL OUTPUT")
print(f"{'─' * 70}")

# Get the underlying bar DataFrame
if USE_CUSUM and final_bar_indices is not None:
    # Subsample bars at CUSUM events
    base_bars = bar_results[best_bar_config_name]["df_bars"]
    df_bars_best = base_bars[final_bar_indices.tolist()]
    print(f"\n▸ df_bars_best: {len(df_bars_best):,} bars (CUSUM-filtered from {best_bar_config_name})")
else:
    df_bars_best = bar_results[final_bar_config_name]["df_bars"]
    print(f"\n▸ df_bars_best: {len(df_bars_best):,} bars ({final_bar_config_name})")

# Build config dict
best_bar_config = {
    "name": final_bar_config_name,
    "base_bar_type": bar_results[best_bar_config_name]["config"]["type"],
    "base_bar_name": best_bar_config_name,
    "base_config": bar_results[best_bar_config_name]["config"],
    "use_cusum": USE_CUSUM,
    "cusum_h_mult": CUSUM_H_MULT,
    "cusum_h": CUSUM_H,
    "n_bars": len(df_bars_best),
    "quality": {
        "acf_1": final_bar_quality.acf_1,
        "abs_acf_1": abs(final_bar_quality.acf_1),
        "lb_stat": final_bar_quality.lb_stat,
        "lb_pval": final_bar_quality.lb_pval,
        "jb_stat": final_bar_quality.jb_stat,
        "ret_std": final_bar_quality.ret_std,
    },
    "median_thresholds": {
        "volume_per_min": MEDIAN_VOLUME_PER_MIN,
        "dollar_per_min": MEDIAN_DOLLAR_PER_MIN,
        "ticks_per_min": MEDIAN_TICKS_PER_MIN,
    },
}

print(f"\n▸ best_bar_config:")
for key, value in best_bar_config.items():
    if isinstance(value, dict):
        print(f"    {key}:")
        for k, v in value.items():
            print(f"      {k}: {v}")
    else:
        print(f"    {key}: {value}")

# Log final selection to MLflow
with mlflow.start_run(run_name="phase1_final_selection"):
    mlflow.log_params({
        "final_config_name": final_bar_config_name,
        "base_bar_type": best_bar_config["base_bar_type"],
        "use_cusum": USE_CUSUM,
        "cusum_h_mult": CUSUM_H_MULT if CUSUM_H_MULT else 0,
    })
    mlflow.log_metrics({
        "final_n_bars": len(df_bars_best),
        "final_abs_acf_1": abs(final_bar_quality.acf_1),
        "final_lb_pval": final_bar_quality.lb_pval,
    })

print(f"\n{'=' * 70}")
print("✓ PHASE 1 COMPLETE")
print(f"{'=' * 70}")
print(f"\n  ★ Selected: {final_bar_config_name}")
print(f"  ★ Bars: {len(df_bars_best):,}")
print(f"  ★ |ACF(1)|: {abs(final_bar_quality.acf_1):.5f}")
print(f"  ★ Ready for Phase 2: Triple Barrier Optimization")
print(f"\n{'=' * 70}")

In [0]:
# PHASE 2A — OHLCV Preparation & Volatility Estimation
# ============================================================
# Extends df_bars_best with log returns and 4 EWMA volatility
# columns.  All operations use native Polars expressions —
# no NumPy extraction needed yet.
#
# Input:  df_bars_best  (8,278 × 7 — Phase 1 output)
# Output: df_bars_vol   (8,278 × 12 — ready for TB sweep)
# ============================================================

from datetime import datetime, timezone

print("=" * 70)
print("PHASE 2A: OHLCV PREPARATION & VOLATILITY ESTIMATION")
print("=" * 70)

# ═══════════════════════════════════════════════════════════════
# 1. COMPUTE LOG RETURNS & EWMA VOLATILITY (Polars-native)
# ═══════════════════════════════════════════════════════════════
# log_ret = ln(close[i]) − ln(close[i−1])
# vol_N   = EWMA σ of log_ret with span N, α = 2/(N+1)
# ═══════════════════════════════════════════════════════════════

VOL_SPANS = [20, 50, 100, 200]
_log_ret = pl.col("close").log().diff()

df_bars_vol = (
    df_bars_best
    .with_columns(
        _log_ret.alias("log_ret"),
        *[_log_ret.ewm_std(span=s, min_samples=2).alias(f"vol_{s}")
          for s in VOL_SPANS],
    )
    .with_columns(
        # Row 0: log_ret is null from diff() → neutral 0.0
        pl.col("log_ret").fill_null(0.0),
        # Rows 0–1: vol is null (need ≥ 2 returns for std) →
        # backward-fill with first valid estimate
        # (minor approximation for 2 of 8,278 bars)
        *[pl.col(f"vol_{s}").backward_fill() for s in VOL_SPANS],
    )
)

print(f"\n▸ df_bars_vol shape: {df_bars_vol.shape[0]:,} × {df_bars_vol.shape[1]}")
print(f"  Columns: {df_bars_vol.columns}")

# ═══════════════════════════════════════════════════════════════
# 2. VALIDATION — Zero Nulls
# ═══════════════════════════════════════════════════════════════

null_counts = df_bars_vol.null_count()
total_nulls = sum(null_counts.row(0))
assert total_nulls == 0, f"Expected zero nulls after fill, got {total_nulls}"
print(f"  Null count: {total_nulls} ✓")

# ═══════════════════════════════════════════════════════════════
# 3. VOLATILITY SUMMARY STATISTICS
# ═══════════════════════════════════════════════════════════════

print(f"\n{'─' * 70}")
print("EWMA VOLATILITY SUMMARY (σ of log returns)")
print(f"{'─' * 70}")
print(f"  {'Span':>6} │ {'Mean':>10} │ {'Median':>10} │ {'Min':>10} │ {'Max':>10} │ {'Range':>8}")
print(f"  {'─'*6}─┼─{'─'*10}─┼─{'─'*10}─┼─{'─'*10}─┼─{'─'*10}─┼─{'─'*8}")

for s in VOL_SPANS:
    vol = df_bars_vol[f"vol_{s}"]
    mn, md, lo, hi = vol.mean(), vol.median(), vol.min(), vol.max()
    ratio = hi / lo if lo > 0 else float("inf")
    print(f"  {s:>6} │ {mn*100:>9.4f}% │ {md*100:>9.4f}% │ {lo*100:>9.4f}% │ {hi*100:>9.4f}% │ {ratio:>6.1f}×")

# ═══════════════════════════════════════════════════════════════
# 4. REGIME DETECTION — Feb 4–6 Drawdown
# ═══════════════════════════════════════════════════════════════

print(f"\n{'─' * 70}")
print("REGIME DETECTION: Feb 4–6 Drawdown")
print(f"{'─' * 70}")

# Peak volatility bar across the entire series
peak_idx = df_bars_vol["vol_20"].arg_max()
peak = df_bars_vol.row(peak_idx, named=True)
print(f"\n▸ Peak vol_20: {peak['vol_20']*100:.4f}% at {peak['bucket']}")
print(f"  Close at peak: ${peak['close']:,.2f}")

# Drawdown window statistics
dd_start = datetime(2026, 2, 4, tzinfo=timezone.utc)
dd_end   = datetime(2026, 2, 7, tzinfo=timezone.utc)

dd_bars = df_bars_vol.filter(pl.col("bucket").is_between(dd_start, dd_end))

if dd_bars.height > 0:
    print(f"\n▸ Drawdown window (Feb 4–6): {dd_bars.height} bars")
    print(f"    vol_20: max {dd_bars['vol_20'].max()*100:.4f}%,  "
          f"mean {dd_bars['vol_20'].mean()*100:.4f}%")
    print(f"    Price:  ${dd_bars['low'].min():,.2f} – ${dd_bars['high'].max():,.2f}")
    dd_pct = (dd_bars["low"].min() - dd_bars["high"].max()) / dd_bars["high"].max() * 100
    print(f"    Max intra-window drawdown: {dd_pct:.1f}%")
else:
    print("\n  ⚠ No bars found in drawdown window")

# ═══════════════════════════════════════════════════════════════
# 5. BARRIER DESIGN CONTEXT
# ═══════════════════════════════════════════════════════════════

ret_std = float(df_bars_vol["log_ret"].std())
avg_close = float(df_bars_vol["close"].mean())
med_range = float(
    ((df_bars_vol["high"] - df_bars_vol["low"]) / df_bars_vol["close"]).median()
)

print(f"\n{'─' * 70}")
print("BARRIER DESIGN CONTEXT")
print(f"{'─' * 70}")
print(f"  Return σ:            {ret_std*100:.4f}%  (${ret_std * avg_close:,.2f})")
print(f"  Median bar range:    {med_range*100:.4f}%  (${med_range * avg_close:,.2f})")
print(f"  Range / σ ratio:     {med_range / ret_std:.2f}×")
print(f"  → Barriers < {med_range / ret_std:.1f}σ resolve from intra-bar noise")
print(f"  → Effective barrier floor: ≥ 2.0σ for signal-carrying labels")

# ── Quick preview ─────────────────────────────────────────────
print(f"\n{'─' * 70}")
print("PREVIEW: df_bars_vol (first 5 rows)")
print(f"{'─' * 70}")
print(df_bars_vol.head(5))

print(f"\n{'=' * 70}")
print("✓ Phase 2A complete — df_bars_vol ready for TB sweep")
print(f"{'=' * 70}")

In [0]:
# PHASE 2B — Triple Barrier Parameter Grid Sweep
# ============================================================
# Sweeps 864 TB configurations using a vectorized wavefront
# algorithm (NumPy).  Outer loop over vol_span (extract arrays
# once per span), inner loop over PT × SL × max_hold.
#
# Algorithm: O(max_hold × n) per config.  At each forward step k,
# ALL unresolved events are processed in a single vectorized
# pass.  Early exit when all events resolve.
#
# Input:  df_bars_vol (Phase 2A — 8,278 × 12)
# Output: sweep_results (list[dict]), tb_label_arrays (dict)
#
# Note: apply_triple_barrier imported from btcusdt_utils.labeling
# ============================================================

import time as _time

print("=" * 70)
print("PHASE 2B: TRIPLE BARRIER PARAMETER GRID SWEEP")
print("=" * 70)

# ═══════════════════════════════════════════════════════════════
# 1. GRID DEFINITION
# ═══════════════════════════════════════════════════════════════

PT_MULTS  = [0.5, 1.0, 1.5, 2.0, 3.0, 5.0]
SL_MULTS  = [0.5, 1.0, 1.5, 2.0, 3.0, 5.0]
MAX_HOLDS = [10, 30, 60, 120, 240, 480]
# VOL_SPANS = [20, 50, 100, 200]  # already defined in Cell 10

TOTAL_CONFIGS = len(PT_MULTS) * len(SL_MULTS) * len(MAX_HOLDS) * len(VOL_SPANS)

print(f"\n▸ Grid: {len(PT_MULTS)} PT × {len(SL_MULTS)} SL × {len(MAX_HOLDS)} hold "
      f"× {len(VOL_SPANS)} vol = {TOTAL_CONFIGS:,} configs")

# Quality thresholds for early skip
MIN_SAMPLES    = 100
MIN_BALANCE    = 0.20
MIN_UNIQUENESS = 0.10

# ═══════════════════════════════════════════════════════════════
# 2. EXTRACT CONSTANT ARRAYS (once, outside all loops)
# ═══════════════════════════════════════════════════════════════

_close = df_bars_vol["close"].to_numpy().astype(np.float64)
_high  = df_bars_vol["high"].to_numpy().astype(np.float64)
_low   = df_bars_vol["low"].to_numpy().astype(np.float64)

print(f"  Extracted close/high/low arrays: {len(_close):,} bars")

# ═══════════════════════════════════════════════════════════════
# 3. MAIN SWEEP
# ═══════════════════════════════════════════════════════════════

sweep_results: list[dict] = []
tb_label_arrays: dict[str, dict] = {}

n_evaluated = 0
n_skip_samples = 0
n_skip_balance = 0
n_skip_uniq = 0
n_skip_nan = 0

t0 = _time.perf_counter()

for vol_span in VOL_SPANS:
    # Extract vol array once per span
    _vol = df_bars_vol[f"vol_{vol_span}"].to_numpy().astype(np.float64)

    for pt_mult in PT_MULTS:
        for sl_mult in SL_MULTS:
            for max_hold in MAX_HOLDS:
                cfg_name = (f"pt{pt_mult}_sl{sl_mult}"
                            f"_h{max_hold}_v{vol_span}")

                labels, touch_bars, touch_types, starts, ends = \
                    apply_triple_barrier(
                        _close, _high, _low, _vol,
                        pt_mult, sl_mult, max_hold,
                    )

                n = len(labels)
                if n < MIN_SAMPLES:
                    n_skip_samples += 1
                    continue

                # Quick balance check (cheap — before uniqueness)
                _, counts = np.unique(labels, return_counts=True)
                bal = (float(counts.min() / counts.sum())
                       if len(counts) > 1 else 0.0)
                if bal < MIN_BALANCE:
                    n_skip_balance += 1
                    continue

                # Full label quality (includes uniqueness)
                lq = compute_label_quality(
                    labels, touch_bars, touch_types,
                    starts, ends, name=cfg_name,
                )

                if lq.avg_uniqueness < MIN_UNIQUENESS:
                    n_skip_uniq += 1
                    continue

                if np.isnan(lq.score):
                    n_skip_nan += 1
                    continue

                sweep_results.append({
                    "config_name":      cfg_name,
                    "pt_mult":          pt_mult,
                    "sl_mult":          sl_mult,
                    "max_hold":         max_hold,
                    "vol_span":         vol_span,
                    "n_samples":        lq.n_samples,
                    "entropy":          lq.entropy,
                    "avg_uniqueness":   lq.avg_uniqueness,
                    "acf_1":            lq.acf_1,
                    "abs_acf_1":        abs(lq.acf_1),
                    "acf_5":            lq.acf_5,
                    "median_touch_bars": lq.median_touch_bars,
                    "vertical_pct":     lq.vertical_pct,
                    "class_balance":    lq.class_balance,
                    "score":            lq.score,
                })

                # Store raw arrays (pruned to top-N after sweep)
                tb_label_arrays[cfg_name] = {
                    "labels":       labels,
                    "touch_bars":   touch_bars,
                    "touch_types":  touch_types,
                    "event_starts": starts,
                    "event_ends":   ends,
                }

                n_evaluated += 1

    elapsed = _time.perf_counter() - t0
    n_skipped = n_skip_samples + n_skip_balance + n_skip_uniq + n_skip_nan
    print(f"  vol_span={vol_span:>3}: "
          f"{n_evaluated:>4} evaluated, {n_skipped:>4} skipped "
          f"[{elapsed:.1f}s]")

total_time = _time.perf_counter() - t0
total_skipped = n_skip_samples + n_skip_balance + n_skip_uniq + n_skip_nan

print(f"\n▸ Sweep complete in {total_time:.1f}s")
print(f"  Evaluated:  {n_evaluated:>4} / {TOTAL_CONFIGS}")
print(f"  Skipped:    {total_skipped:>4}  "
      f"(balance={n_skip_balance}, uniq={n_skip_uniq}, "
      f"samples={n_skip_samples}, nan={n_skip_nan})")

# ═══════════════════════════════════════════════════════════════
# 4. RANK & PRUNE ARRAYS
# ═══════════════════════════════════════════════════════════════

sweep_results.sort(key=lambda r: r["score"], reverse=True)

for i, r in enumerate(sweep_results):
    r["rank"] = i + 1

# Keep arrays only for top configs (free memory)
TOP_N = 20
top_names = {r["config_name"] for r in sweep_results[:TOP_N]}
tb_label_arrays = {
    k: v for k, v in tb_label_arrays.items() if k in top_names
}

print(f"\n▸ Retained arrays for top {TOP_N} configs "
      f"({len(tb_label_arrays)} stored)")

# ═══════════════════════════════════════════════════════════════
# 5. TOP 10 SUMMARY TABLE
# ═══════════════════════════════════════════════════════════════

if sweep_results:
    print(f"\n{'─' * 110}")
    print("TOP 10 CONFIGURATIONS  "
          "(score = uniqueness × entropy × (1 − |ACF₁|))")
    print(f"{'─' * 110}")

    hdr = (f"  {'#':>3} │ {'PT':>3} {'SL':>3} {'Hold':>4} {'Vol':>3} "
           f"│ {'Score':>7} │ {'Uniq':>5} {'Ent':>5} "
           f"│ {'|ACF₁|':>6} {'|ACF₅|':>6} "
           f"│ {'Med.T':>5} {'Vert%':>5} {'Bal':>5} "
           f"│ {'N':>6}")
    sep = (f"  {'─'*3}─┼─{'─'*3}─{'─'*3}─{'─'*4}─{'─'*3}─"
           f"┼─{'─'*7}─┼─{'─'*5}─{'─'*5}─"
           f"┼─{'─'*6}─{'─'*6}─"
           f"┼─{'─'*5}─{'─'*5}─{'─'*5}─"
           f"┼─{'─'*6}")
    print(hdr)
    print(sep)

    for r in sweep_results[:10]:
        print(
            f"  {r['rank']:>3} │ "
            f"{r['pt_mult']:>3.1f} {r['sl_mult']:>3.1f} "
            f"{r['max_hold']:>4} {r['vol_span']:>3} │ "
            f"{r['score']:>7.4f} │ "
            f"{r['avg_uniqueness']:>5.3f} {r['entropy']:>5.3f} │ "
            f"{r['abs_acf_1']:>6.4f} {abs(r['acf_5']):>6.4f} │ "
            f"{r['median_touch_bars']:>5.1f} "
            f"{r['vertical_pct']*100:>4.1f}% "
            f"{r['class_balance']:>5.3f} │ "
            f"{r['n_samples']:>6,}"
        )

    # Quick parameter distribution in top 10
    from collections import Counter
    top10 = sweep_results[:10]
    print(f"\n  Parameter modes in top 10:")
    for param in ["pt_mult", "sl_mult", "max_hold", "vol_span"]:
        ctr = Counter(r[param] for r in top10)
        mode_val, mode_cnt = ctr.most_common(1)[0]
        print(f"    {param:>10}: mode={mode_val} ({mode_cnt}/10), "
              f"unique={len(ctr)}")
else:
    print("\n⚠ No configs passed quality filters!")

# ═══════════════════════════════════════════════════════════════
# 6. MLFLOW LOGGING
# ═══════════════════════════════════════════════════════════════

EXPERIMENT_NAME_P2 = \
    "/Users/aaronng.dev1@gmail.com/BTCUSDT_Phase2_TripleBarrier"
mlflow.set_experiment(EXPERIMENT_NAME_P2)

print(f"\n{'─' * 70}")
print(f"MLFLOW LOGGING → {EXPERIMENT_NAME_P2}")
print(f"{'─' * 70}")

if sweep_results:
    with mlflow.start_run(run_name="phase2_tb_sweep") as parent_run:
        mlflow.log_params({
            "grid_pt_mults":  str(PT_MULTS),
            "grid_sl_mults":  str(SL_MULTS),
            "grid_max_holds": str(MAX_HOLDS),
            "grid_vol_spans": str(VOL_SPANS),
            "total_configs":  TOTAL_CONFIGS,
            "min_balance":    MIN_BALANCE,
            "min_uniqueness": MIN_UNIQUENESS,
        })
        mlflow.log_metrics({
            "n_evaluated":  n_evaluated,
            "n_skipped":    total_skipped,
            "sweep_time_s": round(total_time, 1),
            "best_score":   sweep_results[0]["score"],
        })

        # Nested runs for top configs
        n_nested = min(TOP_N, len(sweep_results))
        for r in sweep_results[:n_nested]:
            with mlflow.start_run(
                run_name=r["config_name"], nested=True,
            ):
                mlflow.log_params({
                    "pt_mult":  r["pt_mult"],
                    "sl_mult":  r["sl_mult"],
                    "max_hold": r["max_hold"],
                    "vol_span": r["vol_span"],
                })
                mlflow.log_metrics({
                    "score":            r["score"],
                    "entropy":          r["entropy"],
                    "avg_uniqueness":   r["avg_uniqueness"],
                    "abs_acf_1":        r["abs_acf_1"],
                    "acf_5":            r["acf_5"],
                    "median_touch_bars": r["median_touch_bars"],
                    "vertical_pct":     r["vertical_pct"],
                    "class_balance":    r["class_balance"],
                    "n_samples":        r["n_samples"],
                })

    print(f"  Parent run:   {parent_run.info.run_id}")
    print(f"  Nested runs:  {n_nested}")

print(f"\n{'=' * 70}")
print("✓ Phase 2B complete — sweep_results ranked, top arrays retained")
print(f"{'=' * 70}")

In [0]:
# PHASE 2C — Rank, Visualize & Export
# ============================================================
# Ranks 295 evaluated TB configs by composite score, builds
# PT×SL heatmap + touch-bar histograms, exports df_labeled_best
# and best_tb_config for Phase 3.
#
# Input:  sweep_results (295 dicts, ranked by score)
#         tb_label_arrays (top 20 configs' raw arrays)
#         df_bars_vol (8,278 × 12)
# Output: df_labeled_best (Polars), best_tb_config (dict)
# ============================================================

print("=" * 70)
print("PHASE 2C: RANK, VISUALIZE & EXPORT")
print("=" * 70)

# ═══════════════════════════════════════════════════════════════
# 1. BUILD RESULTS DATAFRAME (Polars)
# ═══════════════════════════════════════════════════════════════
# All 295 configs already passed Phase 2B quality gates:
#   balance ≥ 0.20, median_touch ≥ 3, uniqueness ≥ 0.10, n ≥ 100
# No additional filtering — rank by score (best-available).
# ═══════════════════════════════════════════════════════════════

df_sweep = pl.DataFrame(sweep_results).sort("score", descending=True)

print(f"\n▸ df_sweep: {df_sweep.shape[0]} configs × {df_sweep.shape[1]} columns")
print(f"  Score:  [{df_sweep['score'].min():.4f}, {df_sweep['score'].max():.4f}]")
print(f"  |ACF₁|: [{df_sweep['abs_acf_1'].min():.4f}, {df_sweep['abs_acf_1'].max():.4f}]")
print(f"  Uniq:   [{df_sweep['avg_uniqueness'].min():.3f}, {df_sweep['avg_uniqueness'].max():.3f}]")

# ═══════════════════════════════════════════════════════════════
# 2. HEATMAP DATA (Polars → lists for Plotly)
# ═══════════════════════════════════════════════════════════════
# Best score per PT × SL cell, marginalized over hold & vol_span.
# ═══════════════════════════════════════════════════════════════

pt_vals = sorted(df_sweep["pt_mult"].unique().to_list())
sl_vals = sorted(df_sweep["sl_mult"].unique().to_list())

z_matrix: list[list[float | None]] = []
annot_text: list[list[str]] = []

for pt in pt_vals:
    z_row: list[float | None] = []
    a_row: list[str] = []
    for sl in sl_vals:
        cell = df_sweep.filter(
            (pl.col("pt_mult") == pt) & (pl.col("sl_mult") == sl)
        )
        if cell.height > 0:
            best_score = float(cell["score"].max())
            z_row.append(best_score)
            a_row.append(f"{best_score:.3f}")
        else:
            z_row.append(None)
            a_row.append("—")
    z_matrix.append(z_row)
    annot_text.append(a_row)

n_filled = sum(1 for row in z_matrix for v in row if v is not None)
print(f"\n▸ Heatmap: {len(pt_vals)}×{len(sl_vals)} grid, "
      f"{n_filled} cells with data, "
      f"{len(pt_vals) * len(sl_vals) - n_filled} empty")

# ═══════════════════════════════════════════════════════════════
# 3. PLOTLY DASHBOARD (1×2: heatmap + touch-bar histogram)
# ═══════════════════════════════════════════════════════════════

fig = dark_fig(
    rows=1, cols=2,
    subplot_titles=(
        "PT × SL Score Heatmap (best across hold/vol)",
        "Bars-Until-Touch Distribution (Top 3)",
    ),
    horizontal_spacing=0.12,
)

# ── (1,1) Heatmap ────────────────────────────────────────────
fig.add_trace(
    go.Heatmap(
        z=z_matrix,
        x=[f"{s:.1f}σ" for s in sl_vals],
        y=[f"{p:.1f}σ" for p in pt_vals],
        text=annot_text,
        texttemplate="%{text}",
        textfont=dict(size=10, color=_TEXT),
        colorscale=[
            [0.0, "#0D1117"],
            [0.25, "#1B3A4B"],
            [0.50, "#58A6FF"],
            [0.75, "#52C41A"],
            [1.0, "#F0883E"],
        ],
        colorbar=dict(
            title=dict(text="Score", font=dict(color=_TEXT)),
            tickfont=dict(color=_TEXT_MUT),
            len=0.9,
        ),
        hovertemplate=(
            "PT=%{y}, SL=%{x}<br>"
            "Best score=%{z:.4f}<extra></extra>"
        ),
    ),
    row=1, col=1,
)

# ── (1,2) Touch-bar histogram (integer-aligned bins) ─────────
top_3_names = [r["config_name"] for r in sweep_results[:3]]
hist_colors = [_BLUE, _GREEN, _ORANGE]

for cfg_name, color in zip(top_3_names, hist_colors):
    if cfg_name in tb_label_arrays:
        tb = tb_label_arrays[cfg_name]["touch_bars"]
        short = (cfg_name
                 .replace("pt", "PT=")
                 .replace("_sl", " SL=")
                 .replace("_h", " h=")
                 .replace("_v", " v="))
        fig.add_trace(
            go.Histogram(
                x=tb,
                opacity=0.60,
                marker_color=color,
                name=short,
                xbins=dict(start=0.5, size=1),
            ),
            row=1, col=2,
        )

# Median touch reference line
fig.add_vline(
    x=3, line_dash="dash", line_color=_RED,
    annotation_text="Median = 3",
    annotation_font=dict(color=_RED, size=10),
    row=1, col=2,
)

fig.update_layout(
    height=500,
    title=dict(
        text="Phase 2C: Triple Barrier Sweep — Score Landscape",
        font=dict(size=16),
    ),
    showlegend=True,
    legend=dict(
        orientation="h",
        yanchor="bottom", y=-0.30,
        xanchor="center", x=0.75,
        font=dict(size=10, color=_TEXT),
    ),
    barmode="overlay",
)

fig.update_xaxes(title_text="SL Multiplier", row=1, col=1)
fig.update_yaxes(title_text="PT Multiplier", row=1, col=1)
fig.update_xaxes(title_text="Bars Until Touch", range=[0, 15], row=1, col=2)
fig.update_yaxes(title_text="Count", row=1, col=2)

fig.show()

# ═══════════════════════════════════════════════════════════════
# 4. TOP 10 TABLE (text — consistent with Cell 11 style)
# ═══════════════════════════════════════════════════════════════

print(f"\n{'─' * 110}")
print("TOP 10 CONFIGURATIONS  "
      "(ranked by score = uniqueness × entropy × (1 − |ACF₁|))")
print(f"{'─' * 110}")

hdr = (f"  {'#':>3} │ {'PT':>3} {'SL':>3} {'Hold':>4} {'Vol':>3} "
       f"│ {'Score':>7} │ {'Uniq':>5} {'Ent':>5} "
       f"│ {'|ACF₁|':>6} {'|ACF₅|':>6} "
       f"│ {'Med.T':>5} {'Vert%':>5} {'Bal':>5} "
       f"│ {'N':>6}")
sep = (f"  {'─'*3}─┼─{'─'*3}─{'─'*3}─{'─'*4}─{'─'*3}─"
       f"┼─{'─'*7}─┼─{'─'*5}─{'─'*5}─"
       f"┼─{'─'*6}─{'─'*6}─"
       f"┼─{'─'*5}─{'─'*5}─{'─'*5}─"
       f"┼─{'─'*6}")
print(hdr)
print(sep)

for r in sweep_results[:10]:
    print(
        f"  {r['rank']:>3} │ "
        f"{r['pt_mult']:>3.1f} {r['sl_mult']:>3.1f} "
        f"{r['max_hold']:>4} {r['vol_span']:>3} │ "
        f"{r['score']:>7.4f} │ "
        f"{r['avg_uniqueness']:>5.3f} {r['entropy']:>5.3f} │ "
        f"{r['abs_acf_1']:>6.4f} {abs(r['acf_5']):>6.4f} │ "
        f"{r['median_touch_bars']:>5.1f} "
        f"{r['vertical_pct']*100:>4.1f}% "
        f"{r['class_balance']:>5.3f} │ "
        f"{r['n_samples']:>6,}"
    )

# ═══════════════════════════════════════════════════════════════
# 5. EXPORT — df_labeled_best
# ═══════════════════════════════════════════════════════════════

print(f"\n{'─' * 70}")
print("EXPORT FOR PHASE 3")
print(f"{'─' * 70}")

best_cfg = sweep_results[0]
best_cfg_name = best_cfg["config_name"]

if best_cfg_name not in tb_label_arrays:
    raise ValueError(
        f"Best config '{best_cfg_name}' not in tb_label_arrays! "
        f"Available: {sorted(tb_label_arrays)}"
    )

best_arrays = tb_label_arrays[best_cfg_name]
n_labels = len(best_arrays["labels"])

df_labeled_best = (
    df_bars_vol
    .head(n_labels)
    .with_columns(
        pl.Series("label",       best_arrays["labels"].astype(int)),
        pl.Series("touch_bars",  best_arrays["touch_bars"]),
        pl.Series("touch_type",  best_arrays["touch_types"].astype(int)),
        pl.Series("event_start", best_arrays["event_starts"]),
        pl.Series("event_end",   best_arrays["event_ends"]),
    )
)

print(f"\n▸ df_labeled_best: {df_labeled_best.shape[0]:,} × "
      f"{df_labeled_best.shape[1]}")

# Label distribution
label_names = {0: "LOSS (SL/Vert)", 1: "WIN  (TP)"}
print(f"\n  Label distribution:")
for row in (df_labeled_best.group_by("label")
            .agg(pl.len().alias("n")).sort("label")
            .iter_rows(named=True)):
    pct = row["n"] / n_labels * 100
    print(f"    {row['label']} {label_names.get(row['label'], '?'):>15}: "
          f"{row['n']:>6,}  ({pct:.1f}%)")

# Touch type distribution
TOUCH_NAMES = {0: "Take-Profit", 1: "Stop-Loss", 2: "Vertical"}
print(f"\n  Touch type distribution:")
for row in (df_labeled_best.group_by("touch_type")
            .agg(pl.len().alias("n")).sort("touch_type")
            .iter_rows(named=True)):
    pct = row["n"] / n_labels * 100
    print(f"    {TOUCH_NAMES.get(row['touch_type'], '?'):>12} "
          f"(type={row['touch_type']}): {row['n']:>6,}  ({pct:.1f}%)")

# ═══════════════════════════════════════════════════════════════
# 6. best_tb_config DICT
# ═══════════════════════════════════════════════════════════════

best_tb_config: dict = {
    "config_name":  best_cfg_name,
    "pt_mult":      best_cfg["pt_mult"],
    "sl_mult":      best_cfg["sl_mult"],
    "max_hold":     best_cfg["max_hold"],
    "vol_span":     best_cfg["vol_span"],
    "n_samples":    best_cfg["n_samples"],
    "quality": {
        "score":            best_cfg["score"],
        "entropy":          best_cfg["entropy"],
        "avg_uniqueness":   best_cfg["avg_uniqueness"],
        "acf_1":            best_cfg["acf_1"],
        "abs_acf_1":        best_cfg["abs_acf_1"],
        "acf_5":            best_cfg["acf_5"],
        "median_touch_bars": best_cfg["median_touch_bars"],
        "vertical_pct":     best_cfg["vertical_pct"],
        "class_balance":    best_cfg["class_balance"],
    },
}

print(f"\n▸ best_tb_config:")
for key, value in best_tb_config.items():
    if isinstance(value, dict):
        print(f"    {key}:")
        for k, v in value.items():
            fmt = f"{v:.6f}" if isinstance(v, float) else str(v)
            print(f"      {k}: {fmt}")
    else:
        print(f"    {key}: {value}")

# ═══════════════════════════════════════════════════════════════
# 7. MLFLOW — Log Final Selection
# ═══════════════════════════════════════════════════════════════

mlflow.set_experiment(EXPERIMENT_NAME_P2)

with mlflow.start_run(run_name="phase2c_final_selection"):
    mlflow.log_params({
        "final_config":   best_cfg_name,
        "final_pt_mult":  best_cfg["pt_mult"],
        "final_sl_mult":  best_cfg["sl_mult"],
        "final_max_hold": best_cfg["max_hold"],
        "final_vol_span": best_cfg["vol_span"],
    })
    mlflow.log_metrics({
        "final_score":          best_cfg["score"],
        "final_entropy":        best_cfg["entropy"],
        "final_avg_uniqueness": best_cfg["avg_uniqueness"],
        "final_abs_acf_1":      best_cfg["abs_acf_1"],
        "final_acf_5":          abs(best_cfg["acf_5"]),
        "final_class_balance":  best_cfg["class_balance"],
        "final_vertical_pct":   best_cfg["vertical_pct"],
        "final_n_labels":       float(n_labels),
    })

print(f"\n▸ MLflow: phase2c_final_selection logged to")
print(f"  {EXPERIMENT_NAME_P2}")

# ═══════════════════════════════════════════════════════════════
# 8. GAP ANALYSIS — Plan Targets vs Actual
# ═══════════════════════════════════════════════════════════════

print(f"\n{'─' * 70}")
print("GAP ANALYSIS: Plan Targets vs Best Config")
print(f"{'─' * 70}")

gaps: list[tuple[str, str, float, bool]] = [
    ("Label entropy",      "> 0.95 bits",
     best_cfg["entropy"],            best_cfg["entropy"] > 0.95),
    ("Avg uniqueness",     "> 0.50",
     best_cfg["avg_uniqueness"],     best_cfg["avg_uniqueness"] > 0.50),
    ("|ACF(1)| of labels", "< 0.10",
     best_cfg["abs_acf_1"],          best_cfg["abs_acf_1"] < 0.10),
    ("|ACF(5)| of labels", "< 0.05",
     abs(best_cfg["acf_5"]),         abs(best_cfg["acf_5"]) < 0.05),
    ("Median touch bars",  "> 5",
     best_cfg["median_touch_bars"],  best_cfg["median_touch_bars"] > 5),
    ("Vertical barrier %", "< 30%",
     best_cfg["vertical_pct"],       best_cfg["vertical_pct"] < 0.30),
    ("Class balance",      "∈ [0.40, 0.60]",
     best_cfg["class_balance"],
     0.40 <= best_cfg["class_balance"] <= 0.60),
]

print(f"\n  {'Metric':<22} │ {'Target':<14} │ {'Actual':<10} │ Status")
print(f"  {'─'*22}─┼─{'─'*14}─┼─{'─'*10}─┼─{'─'*10}")
for name, target, actual_val, passed in gaps:
    if "%" in target:
        actual_str = f"{actual_val*100:.1f}%"
    else:
        actual_str = f"{actual_val:.4f}"
    status = "✓ PASS" if passed else "✗ MISS"
    print(f"  {name:<22} │ {target:<14} │ {actual_str:<10} │ {status}")

n_pass = sum(passed for *_, passed in gaps)
print(f"\n  Scorecard: {n_pass}/{len(gaps)} targets met")

if n_pass < len(gaps):
    print(f"\n▸ Root cause: stride-1 event sampling produces heavy overlap")
    print(f"  • Median touch = {best_cfg['median_touch_bars']:.0f} bars → "
          f"~{int(best_cfg['median_touch_bars'])} concurrent labels per bar")
    print(f"  • avg_uniqueness = {best_cfg['avg_uniqueness']:.3f}  (target: > 0.50)")
    print(f"  • |ACF(1)| = {best_cfg['abs_acf_1']:.3f}  (target: < 0.10)")
    print(f"  • No PT/SL/hold combination can fix this — structural property")
    print(f"  → Phase 5 (sample weighting + sequential bootstrap) is")
    print(f"    MISSION-CRITICAL to compensate for label non-independence")

# ── Preview ──────────────────────────────────────────────────
print(f"\n{'─' * 70}")
print("PREVIEW: df_labeled_best (first 5 rows)")
print(f"{'─' * 70}")
print(df_labeled_best.head(5))

print(f"\n{'=' * 70}")
print("✓ PHASE 2C COMPLETE")
print(f"{'=' * 70}")
print(f"\n  ★ Winner:  {best_cfg_name}")
print(f"  ★ Score:   {best_cfg['score']:.4f}  "
      f"(uniq={best_cfg['avg_uniqueness']:.3f} × "
      f"ent={best_cfg['entropy']:.3f} × "
      f"(1−|ACF₁|)={1 - best_cfg['abs_acf_1']:.3f})")
print(f"  ★ Labels:  {n_labels:,}  "
      f"(TP={int(np.sum(best_arrays['touch_types'] == 0)):,} / "
      f"SL={int(np.sum(best_arrays['touch_types'] == 1)):,} / "
      f"Vert={int(np.sum(best_arrays['touch_types'] == 2)):,})")
print(f"  ★ Export:  df_labeled_best "
      f"({df_labeled_best.shape[0]:,} × {df_labeled_best.shape[1]})")
print(f"  ★ Export:  best_tb_config")
print(f"  ★ Next:    Phase 3 — Feature Stationarity & Fracdiff")
print(f"\n{'=' * 70}")

In [0]:
# PHASE 3A — Compute Technical Indicators on Best Bars
# ============================================================
# Applies the Polars lazy pipeline from btcusdt_utils.features to
# df_bars_best.  Schema-adaptive: skips buy_ratio & vwap_distance
# when source columns are absent.
#
# Input:  df_bars_best (Phase 1 output)
# Output: df_features  (~44 columns)
# ============================================================

from btcusdt_utils.features import (
    FEATURE_COLS, FEATURE_GROUPS, FEATURE_CATEGORIES,
    SKIP_STATIONARITY,
    compute_technical_indicators, validate_features,
)

print("=" * 70)
print("PHASE 3A: COMPUTE TECHNICAL INDICATORS")
print("=" * 70)

# ═════════════════════════════════════════════════════════════
# 1. INPUT SCHEMA CHECK
# ═════════════════════════════════════════════════════════════

print(f"\n▸ Input: df_bars_best — {df_bars_best.shape[0]:,} × {df_bars_best.shape[1]}")
print(f"  Columns: {df_bars_best.columns}")
print(f"  Dtypes:  {dict(zip(df_bars_best.columns, [str(d) for d in df_bars_best.dtypes]))}")

cols_present = set(df_bars_best.columns)
schema_report = {
    "open/high/low/close/volume": all(c in cols_present for c in ("open", "high", "low", "close", "volume")),
    "trade_count": "trade_count" in cols_present,
    "buy_volume": "buy_volume" in cols_present,
    "vwap": "vwap" in cols_present,
    "bucket (datetime)": "bucket" in cols_present,
}

print(f"\n  Schema compatibility:")
for check, present in schema_report.items():
    status = "✓" if present else "✗ (feature skipped)"
    print(f"    {check:<30} {status}")

# ═════════════════════════════════════════════════════════════
# 2. COMPUTE FEATURES (Polars lazy pipeline)
# ═════════════════════════════════════════════════════════════
# Use compact Phase 3 config (matches original 39-feature set windows)
_PHASE3_CONFIG = TechnicalConfig(
    sma_windows=(5, 10, 20, 50),
    ema_spans=(5, 10, 20),
    rsi_periods=(14,),
    macd_configs=((12, 26, 9),),
    bb_periods=(20,),
    atr_periods=(14,),
    return_lookbacks=(1, 5, 10, 30),
    vol_ratio_windows=(10, 20),
    sma_cross_pairs=((5, 20),),
    add_time_features=True,
    add_candle_features=True,
)

print(f"\n▸ Computing technical indicators...")
df_features = compute_technical_indicators(
    df_bars_best,
    config=_PHASE3_CONFIG,
    time_col="bucket",
)

print(f"  Result: {df_features.shape[0]:,} × {df_features.shape[1]}")

# ═════════════════════════════════════════════════════════════
# 3. FEATURE INVENTORY BY GROUP
# ═════════════════════════════════════════════════════════════
# Note: Phase 4 produces stationary-by-construction features
# (sma_dist_5 instead of sma_5). Phase 3 features listed below
# are the configurable-window equivalents.
computed_features = [c for c in df_features.columns if c not in df_bars_best.columns]

print(f"\n▸ Feature inventory: {len(computed_features)} features computed")
print(f"\n  {'Group':<22} │ {'Count':>5} │ Features")
print(f"  {'─' * 22}─┼─{'─' * 5}─┼─{'─' * 40}")

# Group by prefix
_GROUP_PREFIXES = [
    ("A. SMA Distance", "sma_dist"),
    ("B. EMA Distance", "ema_dist"),
    ("C. RSI", "rsi_"),
    ("D. MACD", "macd_"),
    ("E. Bollinger", "bb_"),
    ("F. ATR", "atr_"),
    ("G. Returns", "return_"),
    ("H. Volume", "vol_ratio"),
    ("I. Candle", ("body_", "upper_", "lower_", "is_green", "range_pct")),
    ("J. Time", ("hour_", "minute_")),
    ("K. SMA Cross", "sma_cross"),
]

for group_name, prefix in _GROUP_PREFIXES:
    if isinstance(prefix, str):
        present = [c for c in computed_features if c.startswith(prefix)]
    else:
        present = [c for c in computed_features
                   if any(c.startswith(p) or c == p for p in prefix)]
    if present:
        print(f"  {group_name:<22} │ {len(present):>5} │ {', '.join(present[:5])}")

# ═════════════════════════════════════════════════════════════
# 4. VALIDATE — nulls, NaNs, infinities
# ═════════════════════════════════════════════════════════════

print(f"\n▸ Validating features...")
validation_report = validate_features(df_features, computed_features)

print(f"  Features checked:  {validation_report['n_features']}")
print(f"  First clean index: {validation_report['first_clean_idx']}")
print(f"  Cols with nulls:   {len(validation_report['nulls'])}")
print(f"  Cols with NaNs:    {len(validation_report['nans'])}")
print(f"  Cols with ±inf:    {len(validation_report['infs'])}")

if validation_report['nulls']:
    print(f"\n  Null counts (warm-up period):")
    for col, count in sorted(validation_report['nulls'].items(),
                              key=lambda x: -x[1]):
        print(f"    {col:<20}: {count:>4} nulls")

if validation_report['issues']:
    print(f"\n  ⚠ Issues:")
    for issue in validation_report['issues']:
        print(f"    {issue}")
else:
    print(f"  ✓ No anomalous NaN/inf issues beyond warm-up")

# ═════════════════════════════════════════════════════════════
# 5. QUICK STATS ON KEY FEATURES
# ═════════════════════════════════════════════════════════════

first_clean = validation_report['first_clean_idx'] or 0
df_clean_preview = df_features.slice(first_clean)
n_clean = df_clean_preview.shape[0]

print(f"\n▸ Clean rows: {n_clean:,} (from index {first_clean})")
print(f"  Warm-up rows dropped: {first_clean}")
print(f"  Usable data: {n_clean / df_features.shape[0] * 100:.1f}%")

print(f"\n  Sample ranges (after warm-up):")
for feat in ["sma_dist_5", "rsi_14", "macd_hist_12_26_9",
             "atr_pct_14", "return_1", "vol_ratio_10"]:
    if feat in df_features.columns:
        s = df_clean_preview[feat]
        print(f"    {feat:<20}: [{s.min():.6f}, {s.max():.6f}]  "
              f"μ={s.mean():.6f}  σ={s.std():.6f}")

print(f"\n{'=' * 70}")
print("✓ PHASE 3A COMPLETE")
print(f"{'=' * 70}")
print(f"\n  ★ df_features: {df_features.shape[0]:,} × {df_features.shape[1]}")
print(f"  ★ {len(computed_features)} features computed")
print(f"  ★ First clean index: {first_clean}")
print(f"  ★ validation_report stored for Phase 3B")

In [0]:
# PHASE 3B — ADF/KPSS Stationarity Audit
# ============================================================
# Runs dual ADF + KPSS tests on all testable features.
# Joint interpretation (AFML Ch. 5.2):
#   ADF reject + KPSS fail-to-reject → Stationary
#   ADF fail   + KPSS reject         → Non-stationary → FFD
#   Both reject                       → Trend-stationary
#   Both fail                         → Low power
#
# Input:  df_features (8,278 × 44), validation_report
# Output: df_stationarity, non_stationary_features
# ============================================================

print("=" * 70)
print("PHASE 3B: ADF/KPSS STATIONARITY AUDIT")
print("=" * 70)

# ═══════════════════════════════════════════════════════════════
# 1. PREPARE CLEAN DATA SLICE
# ═══════════════════════════════════════════════════════════════

clean_start: int = validation_report["first_clean_idx"] or 0
df_clean = df_features.slice(clean_start)
n_clean = df_clean.shape[0]

# Identify testable features
testable_features = [
    c for c in computed_features if c not in SKIP_STATIONARITY
]
skipped_names = [
    c for c in computed_features if c in SKIP_STATIONARITY
]

print(f"\n▸ Clean slice: {n_clean:,} rows (from index {clean_start})")
print(f"  Testable features: {len(testable_features)}")
print(f"  Skipped (binary/cyclical): {skipped_names}")

# ═══════════════════════════════════════════════════════════════
# 2. RUN DUAL ADF + KPSS TESTS
# ═══════════════════════════════════════════════════════════════

import time as _time

stationarity_results: list[dict] = []
t0 = _time.perf_counter()

for i, feat in enumerate(testable_features):
    series = df_clean[feat].to_numpy().astype(np.float64)
    # Remove any residual NaN/inf
    valid = series[np.isfinite(series)]
    if len(valid) < 100:
        print(f"  ⚠ {feat}: only {len(valid)} valid values — skipping")
        continue

    adf = adf_test(valid, name=feat)
    kps = kpss_test(valid, name=feat)

    # Joint classification
    adf_rejects = adf["stationary"]          # p < 0.01
    kpss_rejects = not kps["stationary"]     # p ≤ 0.05

    if adf_rejects and not kpss_rejects:
        conclusion = "Stationary"
        needs_ffd = False
    elif not adf_rejects and kpss_rejects:
        conclusion = "Non-stationary"
        needs_ffd = True
    elif adf_rejects and kpss_rejects:
        conclusion = "Trend-stationary"
        needs_ffd = True   # conservative: apply FFD
    else:
        conclusion = "Low-power"
        needs_ffd = True   # conservative: treat as non-stationary

    # Get category info from FEATURE_CATEGORIES
    cat_info = FEATURE_CATEGORIES.get(feat, ("—", "—"))

    stationarity_results.append({
        "feature": feat,
        "group": cat_info[0],
        "dtype_annotation": cat_info[1],
        "adf_stat": round(adf["adf_stat"], 4),
        "adf_pval": round(adf["p_value"], 6),
        "adf_stationary": adf_rejects,
        "kpss_stat": round(kps["kpss_stat"], 4),
        "kpss_pval": round(kps["p_value"], 4),
        "kpss_stationary": kps["stationary"],
        "conclusion": conclusion,
        "needs_ffd": needs_ffd,
    })

elapsed = _time.perf_counter() - t0
print(f"\n▸ Tests completed in {elapsed:.1f}s — {len(stationarity_results)} features tested")

# ═══════════════════════════════════════════════════════════════
# 3. BUILD RESULTS DATAFRAME
# ═══════════════════════════════════════════════════════════════

df_stationarity = pl.DataFrame(stationarity_results)

# Classify lists
stationary_features = (
    df_stationarity
    .filter(pl.col("needs_ffd").not_())
    ["feature"].to_list()
)
non_stationary_features = (
    df_stationarity
    .filter(pl.col("needs_ffd"))
    ["feature"].to_list()
)

# ═══════════════════════════════════════════════════════════════
# 4. DETAILED REPORT BY GROUP
# ═══════════════════════════════════════════════════════════════

print(f"\n{'─' * 110}")
print("STATIONARITY AUDIT RESULTS")
print(f"{'─' * 110}")

hdr = (f"  {'Feature':<18} │ {'Group':<14} │ "
       f"{'ADF stat':>9} {'p-val':>8} {'ADF?':>5} │ "
       f"{'KPSS stat':>9} {'p-val':>7} {'KPSS?':>5} │ "
       f"{'Conclusion':<18}")
print(hdr)
print(f"  {'─' * 18}─┼─{'─' * 14}─┼─"
      f"{'─' * 9}─{'─' * 8}─{'─' * 5}─┼─"
      f"{'─' * 9}─{'─' * 7}─{'─' * 5}─┼─"
      f"{'─' * 18}")

# Sort by group then feature for clean display
for row in df_stationarity.sort("group", "feature").iter_rows(named=True):
    adf_mark = "✓" if row["adf_stationary"] else "✗"
    kpss_mark = "✓" if row["kpss_stationary"] else "✗"

    # Status indicator
    if row["conclusion"] == "Stationary":
        tag = "✓ Stationary"
    elif row["conclusion"] == "Non-stationary":
        tag = "✗ Non-stationary"
    elif row["conclusion"] == "Trend-stationary":
        tag = "⚠ Trend-stat."
    else:
        tag = "⚠ Low-power"

    print(
        f"  {row['feature']:<18} │ {row['group']:<14} │ "
        f"{row['adf_stat']:>9.4f} {row['adf_pval']:>8.6f} {adf_mark:>5} │ "
        f"{row['kpss_stat']:>9.4f} {row['kpss_pval']:>7.4f} {kpss_mark:>5} │ "
        f"{tag:<18}"
    )

# ═══════════════════════════════════════════════════════════════
# 5. SUMMARY SCORECARD
# ═══════════════════════════════════════════════════════════════

counts = (
    df_stationarity
    .group_by("conclusion")
    .agg(pl.len().alias("n"))
    .sort("conclusion")
)

print(f"\n{'─' * 70}")
print("SUMMARY")
print(f"{'─' * 70}")
for row in counts.iter_rows(named=True):
    print(f"  {row['conclusion']:<20}: {row['n']:>3} features")
print(f"  {'─' * 30}")
print(f"  {'Total testable':<20}: {len(stationarity_results):>3} features")
print(f"  {'Skipped (exempt)':<20}: {len(skipped_names):>3} features")

print(f"\n▸ Features needing FFD ({len(non_stationary_features)}):")
for feat in non_stationary_features:
    cat = FEATURE_CATEGORIES.get(feat, ("—", "—"))
    print(f"    {feat:<18} ({cat[0]}, {cat[1]})")

print(f"\n▸ Already stationary ({len(stationary_features)}):")
for feat in stationary_features:
    print(f"    {feat}")

print(f"\n{'=' * 70}")
print("✓ PHASE 3B COMPLETE")
print(f"{'=' * 70}")
print(f"\n  ★ {len(stationary_features)} features stationary (d=0)")
print(f"  ★ {len(non_stationary_features)} features need FFD treatment")
print(f"  ★ df_stationarity ({df_stationarity.shape[0]} × {df_stationarity.shape[1]}) ready")
print(f"  ★ non_stationary_features list exported for Cell 15")

In [0]:
# PHASE 3C — FFD Minimum-d Search & Transform
# ============================================================
# For each non-stationary feature (17 total from Phase 3B):
#   1. find_min_ffd() sweeps d ∈ [0, 1] step 0.05
#   2. Finds minimum d* where ADF p < 0.01 AND corr > 0.90
#   3. Fallback: relax corr to 0.85, then d=1.0 last resort
#   4. Applies FFD at d* to transform the feature
#   5. Verifies post-FFD stationarity
#
# Input:  df_features, non_stationary_features, df_clean
# Output: fracdiff_map, df_ffd_sweep, df_features_stationary
# ============================================================

import time as _time

print("=" * 70)
print("PHASE 3C: FFD MINIMUM-d SEARCH & TRANSFORM")
print("=" * 70)

# ═══════════════════════════════════════════════════════════════
# 1. FFD SWEEP PER FEATURE
# ═══════════════════════════════════════════════════════════════

ADF_PVAL_THRESHOLD = 0.01
CORR_PRIMARY = 0.90
CORR_FALLBACK = 0.85

fracdiff_map: dict[str, float] = {}     # feature → d*
ffd_sweep_rows: list[dict] = []          # full sweep data for viz
ffd_summary: list[dict] = []             # one row per feature

print(f"\n▸ Sweeping d ∈ [0.0, 1.0] for {len(non_stationary_features)} features...")
print(f"  Primary: ADF p < {ADF_PVAL_THRESHOLD}, corr > {CORR_PRIMARY}")
print(f"  Fallback: relax corr to {CORR_FALLBACK}")

t0 = _time.perf_counter()

for i, feat in enumerate(non_stationary_features):
    series = df_clean[feat].to_numpy().astype(np.float64)
    valid = series[np.isfinite(series)]

    # Primary sweep: corr > 0.90
    result = find_min_ffd(
        valid,
        adf_pval=ADF_PVAL_THRESHOLD,
        min_corr=CORR_PRIMARY,
    )

    # Store full sweep data for Cell 16 visualization
    for entry in result["sweep"]:
        ffd_sweep_rows.append({"feature": feat, **entry})

    d_star = result["optimal_d"]
    fallback_used = False

    # Fallback: relax correlation threshold
    if d_star is None:
        for entry in result["sweep"]:
            if entry["stationary"] and entry["corr"] >= CORR_FALLBACK:
                d_star = entry["d"]
                fallback_used = True
                break

    # Last resort: d = 1.0
    if d_star is None:
        d_star = 1.0
        fallback_used = True

    fracdiff_map[feat] = d_star

    # Find correlation at d*
    corr_at_dstar = None
    for entry in result["sweep"]:
        if abs(entry["d"] - d_star) < 1e-6:
            corr_at_dstar = entry["corr"]
            break

    status = "✓" if not fallback_used else ("⚠ relaxed" if d_star < 1.0 else "✗ d=1.0")
    print(f"  [{i+1:>2}/{len(non_stationary_features)}] {feat:<18}  "
          f"d*={d_star:.2f}  corr={corr_at_dstar or 0:.4f}  {status}")

    ffd_summary.append({
        "feature": feat,
        "d_star": d_star,
        "corr_at_dstar": corr_at_dstar,
        "fallback_used": fallback_used,
        "group": FEATURE_CATEGORIES.get(feat, ("—",))[0],
    })

elapsed = _time.perf_counter() - t0
print(f"\n▸ Sweep completed in {elapsed:.1f}s")

# Build sweep DataFrame for Cell 16
df_ffd_sweep = pl.DataFrame(ffd_sweep_rows)
print(f"  df_ffd_sweep: {df_ffd_sweep.shape[0]:,} rows "
      f"({len(non_stationary_features)} features × ~21 d-values)")

# ═══════════════════════════════════════════════════════════════
# 2. APPLY FFD TRANSFORMS
# ═══════════════════════════════════════════════════════════════
# Replace non-stationary columns in-place (same names → no
# downstream rename cascades).
# ═══════════════════════════════════════════════════════════════

print(f"\n▸ Applying FFD transforms...")

df_features_stationary = df_clean.clone()

for feat, d_star in fracdiff_map.items():
    if d_star == 0.0:
        continue  # no transform needed

    original = df_clean[feat].to_numpy().astype(np.float64)
    transformed = ffd(original, d_star)
    df_features_stationary = df_features_stationary.with_columns(
        pl.Series(feat, transformed)
    )

# Find first row where ALL features are valid (no NaN from FFD)
all_feature_cols = [c for c in computed_features if c in df_features_stationary.columns]
first_valid_idx: int | None = None
for idx in range(df_features_stationary.shape[0]):
    row = df_features_stationary.row(idx, named=True)
    if all(
        row[c] is not None and np.isfinite(row[c])
        for c in all_feature_cols
        if df_features_stationary[c].dtype.is_float()
    ):
        first_valid_idx = idx
        break

if first_valid_idx is not None and first_valid_idx > 0:
    df_features_stationary = df_features_stationary.slice(first_valid_idx)

print(f"  FFD NaN trim: dropped {first_valid_idx or 0} leading rows")
print(f"  df_features_stationary: {df_features_stationary.shape[0]:,} × "
      f"{df_features_stationary.shape[1]}")

# Also include d=0 for stationary features in the full map
for feat in stationary_features:
    fracdiff_map[feat] = 0.0

# ═══════════════════════════════════════════════════════════════
# 3. VERIFY POST-FFD STATIONARITY
# ═══════════════════════════════════════════════════════════════

print(f"\n▸ Verifying post-FFD stationarity (ADF re-test)...")

verification_results: list[dict] = []
n_pass = 0
n_fail = 0

for feat in non_stationary_features:
    d_star = fracdiff_map[feat]
    series = df_features_stationary[feat].to_numpy().astype(np.float64)
    valid = series[np.isfinite(series)]

    if len(valid) < 50:
        print(f"  ⚠ {feat}: insufficient valid data ({len(valid)})")
        verification_results.append({
            "feature": feat, "d_star": d_star,
            "post_adf_pval": None, "post_stationary": False,
        })
        n_fail += 1
        continue

    post_adf = adf_test(valid, name=feat)
    passed = post_adf["stationary"]

    verification_results.append({
        "feature": feat,
        "d_star": d_star,
        "post_adf_stat": round(post_adf["adf_stat"], 4),
        "post_adf_pval": round(post_adf["p_value"], 8),
        "post_stationary": passed,
    })

    status = "✓" if passed else "✗ FAIL"
    if passed:
        n_pass += 1
    else:
        n_fail += 1

print(f"  Verification: {n_pass}/{len(non_stationary_features)} features "
      f"confirmed stationary")
if n_fail > 0:
    print(f"  ⚠ {n_fail} features did not achieve stationarity")
    for vr in verification_results:
        if not vr["post_stationary"]:
            print(f"    {vr['feature']}: d*={vr['d_star']}, "
                  f"ADF p={vr.get('post_adf_pval', 'N/A')}")

# ═══════════════════════════════════════════════════════════════
# 4. MLFLOW LOGGING
# ═══════════════════════════════════════════════════════════════

EXPERIMENT_NAME_P3 = (
    f"/Users/aaronng.dev1@gmail.com/BTCUSDT_Phase3_Stationarity"
)
mlflow.set_experiment(EXPERIMENT_NAME_P3)

with mlflow.start_run(run_name="phase3c_ffd_search"):
    # Log fracdiff map as params
    for feat, d_val in fracdiff_map.items():
        safe_key = feat.replace(".", "_")[:250]
        mlflow.log_param(f"d_{safe_key}", d_val)

    # Log summary metrics
    ffd_features = [f for f, d in fracdiff_map.items() if d > 0]
    mean_d = np.mean([d for d in fracdiff_map.values() if d > 0]) if ffd_features else 0.0
    mean_corr = np.mean([s["corr_at_dstar"] for s in ffd_summary
                         if s["corr_at_dstar"] is not None]) if ffd_summary else 0.0

    mlflow.log_metrics({
        "n_stationary": float(len(stationary_features)),
        "n_ffd_applied": float(len(ffd_features)),
        "n_total_testable": float(len(stationarity_results)),
        "mean_optimal_d": float(mean_d),
        "mean_corr_at_dstar": float(mean_corr),
        "n_post_ffd_pass": float(n_pass),
        "n_post_ffd_fail": float(n_fail),
        "n_clean_rows": float(df_features_stationary.shape[0]),
    })

print(f"\n▸ MLflow: phase3c_ffd_search logged to")
print(f"  {EXPERIMENT_NAME_P3}")

# ═══════════════════════════════════════════════════════════════
# 5. SUMMARY TABLE
# ═══════════════════════════════════════════════════════════════

print(f"\n{'─' * 80}")
print("FFD SUMMARY")
print(f"{'─' * 80}")
print(f"  {'Feature':<18} │ {'Group':<14} │ {'d*':>5} │ {'Corr':>6} │ {'Fallback':>8} │ Post-ADF")
print(f"  {'─' * 18}─┼─{'─' * 14}─┼─{'─' * 5}─┼─{'─' * 6}─┼─{'─' * 8}─┼─{'─' * 8}")

for s, v in zip(ffd_summary, verification_results):
    corr_str = f"{s['corr_at_dstar']:.4f}" if s['corr_at_dstar'] else "  N/A"
    fb_str = "yes" if s["fallback_used"] else "no"
    post_str = "✓ pass" if v["post_stationary"] else "✗ fail"
    print(f"  {s['feature']:<18} │ {s['group']:<14} │ {s['d_star']:>5.2f} │ {corr_str:>6} │ {fb_str:>8} │ {post_str}")

print(f"\n  Mean d* (FFD features): {mean_d:.3f}")
print(f"  Mean correlation at d*: {mean_corr:.4f}")

print(f"\n{'=' * 70}")
print("✓ PHASE 3C COMPLETE")
print(f"{'=' * 70}")
print(f"\n  ★ fracdiff_map: {len(fracdiff_map)} entries "
      f"({len(ffd_features)} with d>0, "
      f"{len(stationary_features)} with d=0)")
print(f"  ★ df_features_stationary: {df_features_stationary.shape[0]:,} × "
      f"{df_features_stationary.shape[1]}")
print(f"  ★ df_ffd_sweep: {df_ffd_sweep.shape[0]:,} rows (for Cell 16 viz)")
print(f"  ★ Verification: {n_pass}/{len(non_stationary_features)} "
      f"post-FFD stationary")

In [0]:
# PHASE 3D — Fracdiff Visualization & Export
# ============================================================
# 2×2 Plotly dark-theme dashboard:
#   (1,1) FFD sweep heatmap (ADF p-value by feature × d)
#   (1,2) Memory preservation (correlation vs d)
#   (2,1) Before/after overlay (raw vs FFD)
#   (2,2) Stationarity summary by status
#
# Input:  df_ffd_sweep, ffd_summary, df_features, df_features_stationary
# Output: Final exports for Phase 4
# ============================================================

print("=" * 70)
print("PHASE 3D: FRACDIFF VISUALIZATION & EXPORT")
print("=" * 70)

# ═════════════════════════════════════════════════════════════
# 1. PREP DATA FOR HEATMAP
# ═════════════════════════════════════════════════════════════

# Only show features that actually needed FFD (d* > 0)
ffd_active = [s for s in ffd_summary if s["d_star"] > 0]
ffd_active_names = [s["feature"] for s in ffd_active]

# Pivot: feature × d → ADF p-value (for heatmap)
d_values = sorted(df_ffd_sweep["d"].unique().to_list())

# Build heatmap matrices
heat_pval: list[list[float | None]] = []
heat_corr: list[list[float | None]] = []

for feat in ffd_active_names:
    pval_row: list[float | None] = []
    corr_row: list[float | None] = []
    for d in d_values:
        cell = df_ffd_sweep.filter(
            (pl.col("feature") == feat) & (pl.col("d") == d)
        )
        if cell.height > 0:
            pval_row.append(float(cell["adf_pval"][0]))
            corr_row.append(float(cell["corr"][0]))
        else:
            pval_row.append(None)
            corr_row.append(None)
    heat_pval.append(pval_row)
    heat_corr.append(corr_row)

print(f"\n▸ Heatmap: {len(ffd_active_names)} features × {len(d_values)} d-values")

# ═════════════════════════════════════════════════════════════
# 2. SELECT EXAMPLE FEATURE FOR OVERLAY
# ═════════════════════════════════════════════════════════════

# Dynamic: pick first FFD feature, or first available feature
_base_cols = {"bucket", "open", "high", "low", "close", "volume", "trade_count"}
if ffd_active_names:
    example_feat = ffd_active_names[0]
else:
    # All stationary-by-construction — pick a representative feature
    _candidates = [c for c in df_clean.columns if c not in _base_cols and c.startswith(("sma_dist", "atr", "rsi"))]
    example_feat = _candidates[0] if _candidates else [c for c in df_clean.columns if c not in _base_cols][0]

d_star_ex = fracdiff_map.get(example_feat, 0.0)
print(f"  Example feature for overlay: {example_feat} (d*={d_star_ex:.2f})")

# ═════════════════════════════════════════════════════════════
# 3. PLOTLY DASHBOARD (2×2)
# ═════════════════════════════════════════════════════════════

_overlay_title = (
    f"Before/After: {example_feat} (d*={d_star_ex:.2f})"
    if d_star_ex > 0
    else f"{example_feat} (already stationary, d*=0)"
)

fig = dark_fig(
    rows=2, cols=2,
    subplot_titles=(
        "ADF p-value by Feature × d (lower = more stationary)",
        "Memory Preservation: Correlation vs d",
        _overlay_title,
        "Stationarity Classification (all features)",
    ),
    vertical_spacing=0.14,
    horizontal_spacing=0.10,
)

# ── (1,1) FFD Sweep Heatmap ─────────────────────────────────────
if ffd_active_names:
    fig.add_trace(
        go.Heatmap(
            z=heat_pval,
            x=[f"{d:.2f}" for d in d_values],
            y=ffd_active_names,
            colorscale=[
                [0.0, _GREEN], [0.01, "#2EA043"],
                [0.05, _ORANGE], [0.20, _RED], [1.0, "#3D0000"],
            ],
            zmin=0, zmax=0.20,
            colorbar=dict(
                title=dict(text="ADF p", font=dict(color=_TEXT)),
                tickfont=dict(color=_TEXT_MUT), len=0.4, y=0.78,
            ),
            hovertemplate="%{y}<br>d=%{x}<br>ADF p=%{z:.4f}<extra></extra>",
        ),
        row=1, col=1,
    )
    for i, s in enumerate(ffd_active):
        fig.add_trace(
            go.Scatter(
                x=[f"{s['d_star']:.2f}"], y=[s["feature"]],
                mode="markers",
                marker=dict(symbol="diamond", size=12, color="white",
                            line=dict(width=2, color=_CYAN)),
                showlegend=False,
                hovertemplate=f"d*={s['d_star']:.2f}<extra>{s['feature']}</extra>",
            ),
            row=1, col=1,
        )
else:
    fig.add_annotation(
        text="All features already stationary (d*=0)",
        xref="x domain", yref="y domain", x=0.5, y=0.5,
        font=dict(size=14, color=_TEXT_MUT), showarrow=False,
        row=1, col=1,
    )

# ── (1,2) Memory Preservation Curves ─────────────────────────────
trace_colors = [_BLUE, _GREEN, _ORANGE, _RED, _PURPLE,
                _CYAN, _BLUE, _GREEN, _ORANGE]

for i, feat in enumerate(ffd_active_names):
    feat_data = df_ffd_sweep.filter(pl.col("feature") == feat).sort("d")
    d_vals = feat_data["d"].to_list()
    corr_vals = feat_data["corr"].to_list()
    d_star = fracdiff_map[feat]
    fig.add_trace(
        go.Scatter(
            x=d_vals, y=corr_vals,
            mode="lines+markers", marker=dict(size=4),
            line=dict(color=trace_colors[i % len(trace_colors)], width=1.5),
            name=feat, legendgroup="corr", showlegend=True,
            hovertemplate=f"{feat}<br>d=%{{x:.2f}}<br>corr=%{{y:.4f}}<extra></extra>",
        ),
        row=1, col=2,
    )

if ffd_active_names:
    fig.add_hline(
        y=0.90, line_dash="dash", line_color=_RED,
        annotation_text="corr = 0.90",
        annotation_font=dict(color=_RED, size=10),
        row=1, col=2,
    )
else:
    # FIX: removed explicit xref/yref — let row/col auto-resolve
    fig.add_annotation(
        text="No FFD transforms needed",
        x=0.5, y=0.5,
        xref="x2 domain", yref="y2 domain",
        font=dict(size=14, color=_TEXT_MUT), showarrow=False,
    )

# ── (2,1) Before/After Overlay ─────────────────────────────────
raw_vals = df_clean[example_feat].to_list()
show_n = min(2000, len(raw_vals))
bar_idx = list(range(len(raw_vals) - show_n, len(raw_vals)))

fig.add_trace(
    go.Scatter(
        x=bar_idx, y=raw_vals[-show_n:],
        mode="lines", line=dict(color=_BLUE, width=1.5),
        name=f"{example_feat} (raw)", legendgroup="before_after",
        hovertemplate="Bar %{x}<br>Raw=%{y:.4f}<extra></extra>",
    ),
    row=2, col=1,
)

if d_star_ex > 0:
    transformed_vals = ffd(
        df_clean[example_feat].to_numpy().astype(np.float64), d_star_ex,
    )
    # FIX: removed conflicting yaxis="y3" — row/col handles axis routing
    fig.add_trace(
        go.Scatter(
            x=bar_idx, y=list(transformed_vals[-show_n:]),
            mode="lines", line=dict(color=_ORANGE, width=1.5),
            name=f"FFD(d={d_star_ex:.2f})", legendgroup="before_after",
            hovertemplate="Bar %{x}<br>FFD=%{y:.4f}<extra></extra>",
        ),
        row=2, col=1,
    )

# ── (2,2) Stationarity Summary Bar Chart ─────────────────────────
bar_features: list[str] = []
bar_d_values: list[float] = []
bar_colors: list[str] = []

for feat in sorted(stationary_features):
    bar_features.append(feat)
    bar_d_values.append(0.0)
    bar_colors.append(_GREEN)

for s in ffd_summary:
    if s["d_star"] == 0.0:
        bar_features.append(s["feature"])
        bar_d_values.append(0.0)
        bar_colors.append(_CYAN)

for s in sorted(ffd_active, key=lambda x: x["d_star"]):
    bar_features.append(s["feature"])
    bar_d_values.append(s["d_star"])
    bar_colors.append(_BLUE if s["d_star"] <= 0.10 else _ORANGE)

fig.add_trace(
    go.Bar(
        y=bar_features, x=bar_d_values, orientation="h",
        marker_color=bar_colors, showlegend=False,
        hovertemplate="%{y}: d*=%{x:.2f}<extra></extra>",
    ),
    row=2, col=2,
)
fig.add_vline(x=0.01, line_dash="dot", line_color=_TEXT_MUT, row=2, col=2)

# ── Layout ─────────────────────────────────────────────────────
fig.update_layout(
    height=900,
    title=dict(text="Phase 3: Feature Stationarity & Fractional Differentiation", font=dict(size=16)),
    showlegend=True,
    legend=dict(orientation="h", yanchor="bottom", y=-0.12, xanchor="center", x=0.5,
                font=dict(size=9, color=_TEXT)),
)
fig.update_xaxes(title_text="d (differentiation order)", row=1, col=1)
fig.update_yaxes(title_text="Feature", tickfont=dict(size=9), row=1, col=1)
fig.update_xaxes(title_text="d", row=1, col=2)
fig.update_yaxes(title_text="Correlation with original", row=1, col=2)
fig.update_xaxes(title_text="Bar Index", row=2, col=1)
fig.update_yaxes(title_text="Value", row=2, col=1)
fig.update_xaxes(title_text="Optimal d*", row=2, col=2)
fig.update_yaxes(tickfont=dict(size=8), row=2, col=2)

fig.show()

# ═════════════════════════════════════════════════════════════
# 4. PHASE 3 SCORECARD
# ═════════════════════════════════════════════════════════════

print(f"\n{'─' * 70}")
print("PHASE 3 SCORECARD")
print(f"{'─' * 70}")

n_ffd_actual = len([d for d in fracdiff_map.values() if d > 0])
n_stationary_total = len(stationary_features)
n_trend_stat = len([s for s in ffd_summary if s["d_star"] == 0.0])

scorecard = [
    ("All features tested", f"{len(fracdiff_map)}/{len(fracdiff_map)}", True),
    ("Stationary (d=0, no transform)", f"{n_stationary_total}", n_stationary_total > 0),
    ("Trend-stat. (ADF passed at d=0)", f"{n_trend_stat}", True),
    ("FFD-transformed (d>0)", f"{n_ffd_actual}", True),
    ("Mean d* (FFD features only)", f"{mean_d:.3f}", mean_d < 0.50),
    ("Mean correlation at d*", f"{float(mean_corr):.4f}", float(mean_corr) > 0.90),
    ("Post-FFD verification", f"{n_pass}/{len(non_stationary_features)} pass",
     n_pass >= len(non_stationary_features) - 1),
    ("Clean rows retained",
     f"{df_features_stationary.shape[0]:,}/{df_clean.shape[0]:,} "
     f"({df_features_stationary.shape[0]/df_clean.shape[0]*100:.0f}%)",
     df_features_stationary.shape[0] >= 3000),
    ("No fallback needed", f"{'yes' if not any(s['fallback_used'] for s in ffd_summary) else 'no'}",
     not any(s['fallback_used'] for s in ffd_summary)),
]

for name, value, passed in scorecard:
    status = "✓" if passed else "✗"
    print(f"  {status} {name:<35}: {value}")

# ═════════════════════════════════════════════════════════════
# 5. DATA LOSS ANALYSIS
# ═════════════════════════════════════════════════════════════

data_loss_pct = (1 - df_features_stationary.shape[0] / df_clean.shape[0]) * 100

print(f"\n{'─' * 70}")
print("DATA LOSS ANALYSIS")
print(f"{'─' * 70}")
print(f"  Original bars (Phase 1):    {df_bars_best.shape[0]:>8,}")
print(f"  After warm-up trim:         {df_clean.shape[0]:>8,}  "
      f"(lost {df_bars_best.shape[0] - df_clean.shape[0]} from SMA_50)")
print(f"  After FFD NaN trim:         {df_features_stationary.shape[0]:>8,}  "
      f"(lost {df_clean.shape[0] - df_features_stationary.shape[0]} from FFD window)")
print(f"  Total data loss:            {data_loss_pct:.1f}%")

# ═════════════════════════════════════════════════════════════
# 6. FINAL EXPORT SUMMARY
# ═════════════════════════════════════════════════════════════

print(f"\n{'=' * 70}")
print("✓ PHASE 3 COMPLETE")
print(f"{'=' * 70}")
print(f"\n  Exports for Phase 4:")
print(f"    ★ fracdiff_map          : dict ({len(fracdiff_map)} features → d*)")
print(f"    ★ df_features_stationary: {df_features_stationary.shape[0]:,} × "
      f"{df_features_stationary.shape[1]} (all features now I(0))")
print(f"    ★ df_stationarity       : {df_stationarity.shape[0]} × "
      f"{df_stationarity.shape[1]} (audit report)")
print(f"    ★ df_ffd_sweep          : {df_ffd_sweep.shape[0]:,} rows (sweep data)")
print(f"\n  fracdiff_map (d > 0 only):")
for feat, d in sorted(fracdiff_map.items(), key=lambda x: -x[1]):
    if d > 0:
        print(f"    {feat:<18}: d* = {d:.2f}")

if not any(d > 0 for d in fracdiff_map.values()):
    print("    (none — all features already stationary at d*=0)")

print(f"\n  Diagnosis addressed:")
print(f"    #1 Non-stationary features → ✓ FFD applied (mean d*={mean_d:.3f})")
print(f"    #4 No stationarity testing  → ✓ ADF/KPSS audit on all features")

In [0]:
# PHASE 4A — Microstructure Feature Engineering (Tier 1)
# ============================================================
# Uses compute_microstructure_features() from btcusdt_utils.features.
# Enriches bars with VPIN, Kyle's Lambda, Amihud, Roll/CS spreads.
#
# Input:  df_ticks (52.5M × 5), df_bars_best (Phase 1)
# Output: df_bars_micro, micro_feature_names
# ============================================================

import time as _time

print("=" * 70)
print("PHASE 4A: MICROSTRUCTURE FEATURE ENGINEERING")
print("=" * 70)

t0 = _time.perf_counter()
df_bars_micro = compute_microstructure_features(df_ticks, df_bars_best)
elapsed = _time.perf_counter() - t0

# Identify feature columns (exclude base OHLCV, index, and raw aggregates)
_NON_FEATURES = (
    set(df_bars_best.columns)
    | {"bar_idx", "log_ret", "delta_price",
       "buy_volume", "sell_volume", "buy_dollars", "sell_dollars",
       "buy_count", "sell_count"}
)
micro_feature_names = [c for c in df_bars_micro.columns if c not in _NON_FEATURES]

print(f"\n  df_bars_micro: {df_bars_micro.shape[0]:,} × {df_bars_micro.shape[1]}")
print(f"  Micro features: {len(micro_feature_names)}")
print(f"  Elapsed: {elapsed:.1f}s")

# ── Stationarity spot check ──────────────────────────────────────────
print(f"\n{'─' * 70}")
print("STATIONARITY SPOT CHECK")
print(f"{'─' * 70}")

for feat in ["kyle_lambda_50", "amihud_100", "roll_spread_20",
             "vpin_50", "oi_20", "dollar_oi_50"]:
    if feat not in df_bars_micro.columns:
        continue
    arr = df_bars_micro[feat].drop_nulls().to_numpy()
    if len(arr) > 100:
        result = adf_test(arr, name=feat)
        print(f"  {feat:<25}: ADF p={result['p_value']:.6f}  "
              f"{'✓' if result['stationary'] else '⚠'}")

print(f"\n{'=' * 70}")
print("✓ PHASE 4A COMPLETE")
print(f"{'=' * 70}")

In [0]:
# PHASE 4B — Entropy, Structural Break & TA Window Sweep
# ============================================================
# Computes Tier 2 (entropy + structural) and Tier 3 (window-swept
# traditional TA) features using btcusdt_utils subpackages.
# All Tier 3 features are stationary-by-construction.
#
# Input:  df_bars_micro (Phase 4A), df_bars_best (Phase 1)
# Output: df_features_all, tier2_feature_names, tier3_feature_names
# ============================================================

import time as _time

print("=" * 70)
print("PHASE 4B: ENTROPY, STRUCTURAL BREAK & TA WINDOW SWEEP")
print("=" * 70)

t_phase = _time.perf_counter()

# ── 1. Extract log returns ─────────────────────────────────────────
log_ret = df_bars_micro["log_ret"].to_numpy().astype(np.float64)
print(f"\n▸ Step 1: Extracting arrays...")
print(f"  Arrays extracted: {len(log_ret):,} bars, {_time.perf_counter() - t_phase:.2f}s")

# ── 2. Tier 2a: Entropy features ───────────────────────────────────
t0 = _time.perf_counter()
entropy_arrays = compute_entropy_features(log_ret)
print(f"\n▸ Step 2: Computing Tier 2 entropy features...")
print(f"  Entropy features: {len(entropy_arrays)} computed in {_time.perf_counter() - t0:.1f}s")

# ── 3. Tier 2b: Structural break features ───────────────────────
t0 = _time.perf_counter()
structural_arrays = compute_structural_features(log_ret)
tier2_feature_names = list(entropy_arrays) + list(structural_arrays)
_cusum_ct = sum(1 for k in structural_arrays if "cusum" in k)
_adf_ct = sum(1 for k in structural_arrays if "adf_stat" in k)
print(f"\n▸ Step 3: Computing Tier 2 structural break features...")
print(f"  CUSUM features: {_cusum_ct} computed")
print(f"  Computing rolling ADF statistics (may take ~15s)...")
print(f"  Rolling ADF: {_adf_ct} features in {_time.perf_counter() - t0:.1f}s")
print(f"\n  Total Tier 2 features: {len(tier2_feature_names)} in {_time.perf_counter() - t0:.1f}s")

# ── 4. Tier 3: TA window sweep (stationary-by-construction) ──
t0 = _time.perf_counter()
df_ta = compute_technical_indicators(df_bars_best, config=TechnicalConfig())
tier3_feature_names = [c for c in df_ta.columns if c not in df_bars_best.columns]
print(f"\n▸ Step 4: Computing Tier 3 traditional TA sweep...")
print(f"\n  Total Tier 3 features: {len(tier3_feature_names)} in {_time.perf_counter() - t0:.1f}s")

# ── 5. Assemble df_features_all ───────────────────────────────────
t0 = _time.perf_counter()
tier2_cols = [pl.Series(k, v) for k, v in {**entropy_arrays, **structural_arrays}.items()]
df_features_all = df_bars_micro.with_columns(tier2_cols).hstack(
    df_ta.select(tier3_feature_names)
)

# Drop features with 0 valid (non-null AND non-NaN) values
_float_cols = [
    c for c in df_features_all.columns
    if df_features_all[c].dtype in (pl.Float32, pl.Float64)
]

def _valid_count(col_name: str) -> int:
    """Count of non-null, non-NaN values."""
    s = df_features_all[col_name]
    return int(s.drop_nulls().filter(~s.drop_nulls().is_nan()).len())

_zero_valid = [c for c in _float_cols if _valid_count(c) == 0]
if _zero_valid:
    print(f"  Dropped {len(_zero_valid)} features (0 valid values)")
    df_features_all = df_features_all.drop(_zero_valid)
    micro_feature_names = [c for c in micro_feature_names if c not in _zero_valid]

print(f"\n▸ Step 5: Assembling df_features_all...")
print(f"  df_features_all: {df_features_all.shape[0]:,} × {df_features_all.shape[1]}")
print(f"  Assembly time: {_time.perf_counter() - t0:.2f}s")

# ── 6. Stationarity audit ──────────────────────────────────────────
t0 = _time.perf_counter()
print(f"\n▸ Step 6: Stationarity audit...")
print(f"  {'Feature':<30} {'ADF p':>8}  Status")
print(f"  {'─' * 24} {'─' * 10}  {'─' * 8}")

_all_stat = True
for feat in ["sma_dist_50", "ema_dist_100", "rsi_14", "macd_hist_12_26_9",
             "bb_pct_20", "return_5", "shannon_50", "cusum_plus_raw",
             "adf_stat_100", "martingale_ratio_50", "lz_complexity_50"]:
    if feat not in df_features_all.columns:
        continue
    arr = df_features_all[feat].drop_nulls().to_numpy()
    if len(arr) > 100:
        res = adf_test(arr, name=feat)
        status = "✓" if res["stationary"] else "⚠"
        if not res["stationary"]:
            _all_stat = False
        print(f"  {feat:<30} {res['p_value']:.6f}  {status}")

print(f"\n  {'✓ All audited features are stationary' if _all_stat else '⚠ Some features non-stationary'}")
print(f"  Audit time: {_time.perf_counter() - t0:.1f}s")

In [0]:
# PHASE 4B-2 — MI Scoring & Feature Evaluation
# ============================================================
# Scores all features via MI + permutation noise floor + single-
# feature LightGBM AUC.  Uses btcusdt_utils.scoring subpackage.
#
# Input:  df_features_all (4B), df_labeled_best (Phase 2)
# Output: df_mi_sweep (Polars, scored features)
# ============================================================

import time as _time

print("\n▸ Step 7: Label alignment...")
t0 = _time.perf_counter()

# Join features with optimized TB labels
df_aligned = df_features_all.join(
    df_labeled_best.select(["bucket", "label"]), on="bucket", how="inner",
)

# Build feature candidate list (exclude non-features)
_EXCLUDE = {
    "bar_idx", "bucket", "open", "high", "low", "close", "volume",
    "trade_count", "log_ret", "delta_price", "label",
    "buy_volume", "sell_volume", "buy_dollars", "sell_dollars",
    "buy_count", "sell_count",
}
all_feature_names = [c for c in df_aligned.columns if c not in _EXCLUDE]
print(f"  After join: {df_aligned.shape[0]:,} rows with labels")
print(f"  Feature candidates: {len(all_feature_names)}")

# Pre-filter: drop features with >10% NaN/null values
_n_rows = df_aligned.shape[0]
_high_nan = []
for c in all_feature_names:
    s = df_aligned[c]
    _null_ct = s.null_count()
    _nan_ct = int(s.is_nan().sum()) if s.dtype.is_float() else 0
    if (_null_ct + _nan_ct) > _n_rows * 0.10:
        _high_nan.append(c)
if _high_nan:
    print(f"  Dropped {len(_high_nan)} features with >10% missing: {_high_nan}")
    all_feature_names = [c for c in all_feature_names if c not in _high_nan]
    print(f"  Remaining candidates: {len(all_feature_names)}")

# Extract matrices and drop NaN rows
X_full = df_aligned.select(all_feature_names).to_numpy().astype(np.float64)
y_full = df_aligned["label"].to_numpy().astype(np.int32)
mask_valid = np.all(np.isfinite(X_full), axis=1)
X, y = X_full[mask_valid], y_full[mask_valid]
print(f"  After NaN drop: {X.shape[0]:,} usable rows ({X.shape[0]/len(y_full)*100:.1f}%)")
print(f"  Alignment time: {_time.perf_counter() - t0:.2f}s")

assert X.shape[0] > 0, f"No valid rows after NaN drop! Check feature NaN rates."

# ── 8. MI scoring ─────────────────────────────────────────────────
t0 = _time.perf_counter()
mi_scores = compute_mi_scores(X, y)
print(f"\n▸ Step 8: MI scoring...")
print(f"  MI computed for {len(mi_scores)} features in {_time.perf_counter() - t0:.1f}s")

# ── 9. Permutation noise floor ───────────────────────────────────
t0 = _time.perf_counter()
noise = compute_noise_floor(X, y)
print(f"\n▸ Step 9: Permutation noise floor (100 shuffles)...")
print(f"  Noise floor (95th pct): {noise.p95:.6f}")
print(f"  Noise floor (99th pct): {noise.p99:.6f}")
print(f"  Permutation time: {_time.perf_counter() - t0:.1f}s")

# ── 10. LightGBM single-feature AUC ─────────────────────────────
t0 = _time.perf_counter()
auc_scores = compute_univariate_auc(X, y)
print(f"\n▸ Step 10: LightGBM single-feature AUC...")
print(f"  LightGBM AUC computed in {_time.perf_counter() - t0:.1f}s")

# ── 11. Build df_mi_sweep results table ─────────────────────────
print("\n▸ Step 11: Building df_mi_sweep results table...")

_micro_set = set(micro_feature_names)
_t2_set = set(tier2_feature_names)

def _classify(name: str) -> tuple[str, str]:
    """Return (tier, family) for a feature."""
    if name in _micro_set:
        for pfx, fam in [("kyle", "Kyle Lambda"), ("amihud", "Amihud"),
                         ("roll_spread", "Roll Spread"), ("vpin", "VPIN"),
                         ("cs_spread", "Corwin-Schultz")]:
            if pfx in name:
                return "Tier 1", fam
        return "Tier 1", "Order Flow"
    if name in _t2_set:
        if "shannon" in name or "lz_" in name:
            return "Tier 2", "Entropy"
        return "Tier 2", "Structural Break"
    _t3_map = [
        ("sma_dist", "SMA Distance"), ("ema_dist", "EMA Distance"),
        ("sma_cross", "SMA Cross"), ("rsi", "RSI"), ("macd", "MACD"),
        ("bb_", "Bollinger"), ("atr", "ATR"), ("return_", "Returns"),
        ("vol_ratio", "Volume"), ("body", "Candle"), ("shadow", "Candle"),
        ("is_green", "Candle"), ("range_pct", "Candle"),
        ("hour", "Cyclical"), ("minute", "Cyclical"),
    ]
    for pfx, fam in _t3_map:
        if name.startswith(pfx):
            return "Tier 3", fam
    return "Tier 3", "Other"

records = []
for j, feat in enumerate(all_feature_names):
    tier, family = _classify(feat)
    records.append({
        "feature": feat, "tier": tier, "family": family,
        "mi_score": float(mi_scores[j]), "lgb_auc": float(auc_scores[j]),
        "above_noise": bool(mi_scores[j] > noise.p95),
        "mi_ratio": float(mi_scores[j] / noise.p95) if noise.p95 > 0 else 0.0,
    })

df_mi_sweep = pl.DataFrame(records).sort("mi_score", descending=True)

# ── Summary ─────────────────────────────────────────────────────
n_above = df_mi_sweep.filter(pl.col("above_noise") == True).shape[0]
print(f"\n{'=' * 70}")
print("PHASE 4B RESULTS SUMMARY")
print(f"{'=' * 70}")
print(f"\n  ★ Total features scored:  {len(all_feature_names)}")
print(f"  ★ Tier 1 (Microstructure): {df_mi_sweep.filter(pl.col('tier') == 'Tier 1').shape[0]}")
print(f"  ★ Tier 2 (Entropy/Struct): {df_mi_sweep.filter(pl.col('tier') == 'Tier 2').shape[0]}")
print(f"  ★ Tier 3 (Traditional TA): {df_mi_sweep.filter(pl.col('tier') == 'Tier 3').shape[0]}")
print(f"  ★ Features above noise:   {n_above} ({n_above/len(all_feature_names)*100:.1f}%)")
print(f"  ★ Noise floor (95%):      {noise.p95:.6f}")
print(f"  ★ Best MI score:          {mi_scores.max():.6f}")
print(f"  ★ Best LightGBM AUC:      {auc_scores.max():.4f}")
print(f"  ★ Usable samples:         {X.shape[0]:,}")

print(f"\n{'─' * 70}")
print("TOP 20 FEATURES BY MI SCORE")
print(f"{'─' * 70}")
print(f"  {'Rank':<4} {'Feature':<28} {'Tier':<8} {'Family':<16} {'MI':>8} {'AUC':>6} {'Signal'}")
print(f"  {'─'*4} {'─'*28} {'─'*8} {'─'*16} {'─'*8} {'─'*6} {'─'*6}")
for i, row in enumerate(df_mi_sweep.head(20).iter_rows(named=True)):
    sig = "✓" if row["above_noise"] else ""
    print(f"  {i+1:<4} {row['feature']:<28} {row['tier']:<8} {row['family']:<16} "
          f"{row['mi_score']:.6f} {row['lgb_auc']:.4f} {sig}")

print(f"\n{'─' * 70}")
print("FEATURES ABOVE NOISE FLOOR BY FAMILY")
print(f"{'─' * 70}")
df_above = df_mi_sweep.filter(pl.col("above_noise") == True)
if df_above.shape[0] > 0:
    for row in df_above.group_by("family").agg(pl.len().alias("count")).sort("count", descending=True).iter_rows(named=True):
        print(f"  {row['family']:<20}: {row['count']}")
else:
    print("  (No features above noise floor)")

print(f"\n{'=' * 70}")
print("✓ PHASE 4B COMPLETE")
print(f"{'=' * 70}")
print(f"\n  ★ df_mi_sweep:           {df_mi_sweep.shape[0]} features scored")
print(f"  ★ df_features_all:       {df_features_all.shape[0]:,} × {df_features_all.shape[1]}")
print(f"  ★ tier2_feature_names:   {len(tier2_feature_names)}")
print(f"  ★ tier3_feature_names:   {len(tier3_feature_names)}")
print(f"  ★ Next: Phase 5 — Sample Weighting")